# CRVSE Phase 2-10 : Live-Compatible PhysFormer Fine-Tuning

## Research question

Can the current multichannel CRVSE PhysFormer be fine-tuned on live-compatible 8-second rPPG windows so that it performs better in the browser-app workflow?

The goal is not to make the model look good by hiding uncertainty. The goal is to test whether the current checkpoint can adapt to the live-compatible preprocessing distribution.

## Current hypothesis

The existing model is not globally useless. Previous audit work suggests it performs reasonably on stored training-style windows, but live/browser-compatible preprocessing introduces a distribution shift.

The first fine-tuning target is therefore:

- start from the current multichannel PhysFormer checkpoint,
- train only on live-compatible windows from MCD-rPPG, UBFC-rPPG, and UBFC-Phys,
- exclude ECG-Fitness from the first training objective,
- keep ECG-Fitness as out-of-domain evaluation,
- compare against the frozen live-compatible baseline.

## First success criterion

Before training anything, this notebook must reproduce the manifest counts:

```text
finetune_train: 12,503
finetune_val:    2,839
finetune_test:   2,828
ood_eval:          882
total eval:     19,052

## Imports and Setup

In [1]:
import json, math, os, random, sys, h5py, torch, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import resample, butter, filtfilt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle runtime locations
KAGGLE_WORKING = Path("/kaggle/working")
LIVE_COMPATIBLE_DIR = Path("/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE")
RPPG_ENSEMBLE_DIR = Path("/kaggle/input/datasets/cezarytubacki/rppg-ensemble/rPPG ensemble")

# HDF5 ensemble datasets
H5_PATHS = {
    "mcd_rppg": RPPG_ENSEMBLE_DIR / "mcd_rppg_ensemble.h5",
    "ubfc_rppg": RPPG_ENSEMBLE_DIR / "ubfc_rppg_ensemble.h5",
    "ubfc_phys": RPPG_ENSEMBLE_DIR / "ubfc_phys_ensemble.h5",
    "ecg_fitness": RPPG_ENSEMBLE_DIR / "ecg_fitness_ensemble.h5",
}

# Live-compatible audit / fine-tuning artifacts
MANIFEST_PATH = LIVE_COMPATIBLE_DIR / "live_finetune_manifest.csv"
BASELINE_SUMMARY_PATH = LIVE_COMPATIBLE_DIR / "live_finetune_frozen_baseline_summary.csv"
BASELINE_PREDICTIONS_PATH = LIVE_COMPATIBLE_DIR / "live_finetune_frozen_baseline_predictions.csv"
CHECKPOINT_PATH = LIVE_COMPATIBLE_DIR / "CRVSETransformer_Ensemble_physformer_multichannel_best.pt"

# Training should normally run on GPU.
# Set DRY_RUN=True only if you want to test paths/counts on CPU without training.
DRY_RUN = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"h5py: {h5py.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

if DEVICE.type != "cuda" and not DRY_RUN:
    raise RuntimeError(
        "CUDA is not available. Enable a Kaggle GPU accelerator before fine-tuning, "
        "or set DRY_RUN=True for CPU-only artifact checks."
    )

required_paths = {
    "live_compatible_dir": LIVE_COMPATIBLE_DIR,
    "rppg_ensemble_dir": RPPG_ENSEMBLE_DIR,
    "manifest": MANIFEST_PATH,
    "baseline_summary": BASELINE_SUMMARY_PATH,
    "baseline_predictions": BASELINE_PREDICTIONS_PATH,
    "checkpoint": CHECKPOINT_PATH,
    **{f"h5_{name}": path for name, path in H5_PATHS.items()},
}

path_report = pd.DataFrame(
    [
        {
            "artifact": name,
            "exists": path.exists(),
            "path": str(path),
        }
        for name, path in required_paths.items()
    ]
)

missing = path_report.loc[~path_report["exists"], "path"].tolist()

if missing:
    raise FileNotFoundError("Missing required Kaggle artifacts:\n" + "\n".join(missing))

artifact_report = {
    "device": str(DEVICE),
    "seed": SEED,
    "manifest_path": str(MANIFEST_PATH),
    "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
    "baseline_predictions_path": str(BASELINE_PREDICTIONS_PATH),
    "checkpoint_path": str(CHECKPOINT_PATH),
    "h5_paths": {name: str(path) for name, path in H5_PATHS.items()},
}

path_report

Python: 3.12.13
NumPy: 2.0.2
Pandas: 2.3.3
h5py: 3.16.0
PyTorch: 2.10.0+cu128
Device: cuda


,artifact,exists,path
0,live_compatible_dir,True,/kaggle/input/datasets/cezarytubacki/live-comp...
1,rppg_ensemble_dir,True,/kaggle/input/datasets/cezarytubacki/rppg-ense...
2,manifest,True,/kaggle/input/datasets/cezarytubacki/live-comp...
3,baseline_summary,True,/kaggle/input/datasets/cezarytubacki/live-comp...
4,baseline_predictions,True,/kaggle/input/datasets/cezarytubacki/live-comp...
5,checkpoint,True,/kaggle/input/datasets/cezarytubacki/live-comp...
6,h5_mcd_rppg,True,/kaggle/input/datasets/cezarytubacki/rppg-ense...
7,h5_ubfc_rppg,True,/kaggle/input/datasets/cezarytubacki/rppg-ense...
8,h5_ubfc_phys,True,/kaggle/input/datasets/cezarytubacki/rppg-ense...
9,h5_ecg_fitness,True,/kaggle/input/datasets/cezarytubacki/rppg-ense...


##  Manifest Sanity Check


The manifest is the experiment contract:

- `finetune_train` windows are used for optimization.
- `finetune_val` windows are used for model selection.
- `finetune_test` windows are held out for final in-domain evaluation.
- `ood_eval` windows are ECG-Fitness stress-domain evaluation only.
- `excluded` windows are not used for training or evaluation.

Expected counts:

```text
total rows:       19,243
evaluation rows:  19,052
fine-tune rows:   12,503

finetune_train:   12,503
finetune_val:      2,839
finetune_test:     2,828
ood_eval:            882
excluded:            191

In [2]:
def parse_bool(value) -> bool:
    """Parse bool-like values from a CSV loaded by pandas."""
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return str(value).strip().lower() in {"true", "1", "yes", "y"}


manifest = pd.read_csv(MANIFEST_PATH)

manifest["include_in_finetune_bool"] = manifest["include_in_finetune"].map(parse_bool)
manifest["include_in_eval_bool"] = manifest["include_in_eval"].map(parse_bool)

expected_role_counts = {
    "finetune_train": 12503,
    "finetune_val": 2839,
    "finetune_test": 2828,
    "ood_eval": 882,
    "excluded": 191,
}

actual_role_counts = manifest["window_role"].value_counts().sort_index()

manifest_summary = {
    "total_rows": len(manifest),
    "fine_tuning_rows": int(manifest["include_in_finetune_bool"].sum()),
    "evaluation_rows": int(manifest["include_in_eval_bool"].sum()),
    "unique_datasets": sorted(manifest["dataset"].unique().tolist()),
    "unique_roles": sorted(manifest["window_role"].unique().tolist()),
}

role_check = pd.DataFrame(
    [
        {
            "window_role": role,
            "expected": expected,
            "actual": int(actual_role_counts.get(role, 0)),
            "matches": int(actual_role_counts.get(role, 0)) == expected,
        }
        for role, expected in expected_role_counts.items()
    ]
)

dataset_role_counts = (
    manifest
    .groupby(["dataset", "window_role"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["dataset", "window_role"])
)

policy_checks = {
    "ecg_fitness_finetune_rows": int(
        manifest.loc[
            (manifest["dataset"] == "ecg_fitness")
            & manifest["include_in_finetune_bool"]
        ].shape[0]
    ),
    "excluded_eval_rows": int(
        manifest.loc[
            (manifest["window_role"] == "excluded")
            & manifest["include_in_eval_bool"]
        ].shape[0]
    ),
    "excluded_finetune_rows": int(
        manifest.loc[
            (manifest["window_role"] == "excluded")
            & manifest["include_in_finetune_bool"]
        ].shape[0]
    ),
    "eval_not_in_expected_roles": int(
        manifest.loc[
            manifest["include_in_eval_bool"]
            & ~manifest["window_role"].isin(["finetune_train", "finetune_val", "finetune_test", "ood_eval"])
        ].shape[0]
    ),
}

print("Manifest summary:")
print(json.dumps(manifest_summary, indent=2))

print("\nRole count check:")
display(role_check)

print("\nRows by dataset and role:")
display(dataset_role_counts)

print("\nPolicy checks:")
print(json.dumps(policy_checks, indent=2))

if not role_check["matches"].all():
    raise AssertionError("Manifest role counts do not match the expected experiment contract.")

if policy_checks["ecg_fitness_finetune_rows"] != 0:
    raise AssertionError("ECG-Fitness appears in fine-tuning rows. It should be OOD-only for this experiment.")

if policy_checks["excluded_eval_rows"] != 0 or policy_checks["excluded_finetune_rows"] != 0:
    raise AssertionError("Excluded rows leaked into evaluation or fine-tuning.")

if policy_checks["eval_not_in_expected_roles"] != 0:
    raise AssertionError("Evaluation contains unexpected window roles.")

print("\nPASS: Manifest matches the intended live-compatible fine-tuning contract.")

Manifest summary:
{
  "total_rows": 19243,
  "fine_tuning_rows": 12503,
  "evaluation_rows": 19052,
  "unique_datasets": [
    "ecg_fitness",
    "mcd_rppg",
    "ubfc_phys",
    "ubfc_rppg"
  ],
  "unique_roles": [
    "excluded",
    "finetune_test",
    "finetune_train",
    "finetune_val",
    "ood_eval"
  ]
}

Role count check:


,window_role,expected,actual,matches
0,finetune_train,12503,12503,True
1,finetune_val,2839,2839,True
2,finetune_test,2828,2828,True
3,ood_eval,882,882,True
4,excluded,191,191,True



Rows by dataset and role:


,dataset,window_role,rows
0,ecg_fitness,excluded,9
1,ecg_fitness,ood_eval,882
2,mcd_rppg,excluded,85
3,mcd_rppg,finetune_test,2094
4,mcd_rppg,finetune_train,7446
5,mcd_rppg,finetune_val,1743
6,ubfc_phys,excluded,97
7,ubfc_phys,finetune_test,658
8,ubfc_phys,finetune_train,4595
9,ubfc_phys,finetune_val,986



Policy checks:
{
  "ecg_fitness_finetune_rows": 0,
  "excluded_eval_rows": 0,
  "excluded_finetune_rows": 0,
  "eval_not_in_expected_roles": 0
}

PASS: Manifest matches the intended live-compatible fine-tuning contract.


## Frozen Baseline Sanity Check

This section verifies the frozen-model baseline that the fine-tuning experiment must beat.

The baseline prediction file contains the current CRVSE PhysFormer evaluated on the live-compatible manifest using several preprocessing modes.

The most important mode for this experiment is:

```text
training_buffer_source_fps

In [3]:
baseline_summary = pd.read_csv(BASELINE_SUMMARY_PATH)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH, low_memory=False)

expected_modes = [
    "stored_reference",
    "training_buffer_source_fps",
    "training_buffer_sim_15fps",
    "training_buffer_sim_10fps",
]

expected_eval_windows = int(manifest["include_in_eval_bool"].sum())
expected_prediction_rows = expected_eval_windows * len(expected_modes)

numeric_cols = [
    "model_abs_error",
    "spectral_abs_error",
    "spectral_spread_bpm",
    "target_hr_mean",
    "target_hr_range",
    "pos_corr_vs_stored",
    "chrom_corr_vs_stored",
    "green_corr_vs_stored",
]

for col in numeric_cols:
    baseline_predictions[col] = pd.to_numeric(baseline_predictions[col], errors="coerce")


def q90(values: pd.Series) -> float:
    """Return the 90th percentile for a numeric pandas Series."""
    return float(values.quantile(0.90))


def metric_table(data: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Aggregate model and spectral error metrics for grouped baseline rows."""
    table = (
        data
        .groupby(group_cols, dropna=False)
        .agg(
            rows=("model_abs_error", "size"),
            model_mae_mean=("model_abs_error", "mean"),
            model_mae_median=("model_abs_error", "median"),
            model_mae_p90=("model_abs_error", q90),
            spectral_mae_mean=("spectral_abs_error", "mean"),
            spectral_mae_median=("spectral_abs_error", "median"),
            spectral_mae_p90=("spectral_abs_error", q90),
            spectral_spread_mean=("spectral_spread_bpm", "mean"),
            target_hr_mean=("target_hr_mean", "mean"),
            target_hr_range_mean=("target_hr_range", "mean"),
        )
        .reset_index()
    )

    metric_cols = [col for col in table.columns if col not in group_cols + ["rows"]]
    table[metric_cols] = table[metric_cols].round(2)
    return table


prediction_shape_check = {
    "baseline_summary_rows": len(baseline_summary),
    "baseline_prediction_rows": len(baseline_predictions),
    "expected_prediction_rows": expected_prediction_rows,
    "expected_eval_windows": expected_eval_windows,
    "unique_modes": sorted(baseline_predictions["preprocessing_mode"].unique().tolist()),
}

status_counts = (
    baseline_predictions
    .groupby(["preprocessing_mode", "preprocessing_status"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["preprocessing_mode", "preprocessing_status"])
)

mode_counts = (
    baseline_predictions
    .groupby("preprocessing_mode", dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("preprocessing_mode")
)

ok_baseline = baseline_predictions[baseline_predictions["preprocessing_status"] == "ok"].copy()

overall_by_mode = metric_table(ok_baseline, ["preprocessing_mode"]).sort_values("preprocessing_mode")

val_test_by_mode = metric_table(
    ok_baseline[
        ok_baseline["window_role"].isin(["finetune_val", "finetune_test"])
    ],
    ["window_role", "preprocessing_mode"],
).sort_values(["window_role", "preprocessing_mode"])

source_fps_by_dataset_role = metric_table(
    ok_baseline[
        ok_baseline["preprocessing_mode"] == "training_buffer_source_fps"
    ],
    ["dataset", "window_role"],
).sort_values(["window_role", "dataset"])

source_fps_targets = (
    val_test_by_mode[
        val_test_by_mode["preprocessing_mode"] == "training_buffer_source_fps"
    ][
        [
            "window_role",
            "rows",
            "model_mae_mean",
            "model_mae_median",
            "model_mae_p90",
            "spectral_mae_mean",
            "spectral_mae_median",
            "spectral_mae_p90",
        ]
    ]
    .reset_index(drop=True)
)

print("Prediction shape check:")
print(json.dumps(prediction_shape_check, indent=2))

print("\nStatus counts:")
display(status_counts)

print("\nRows per preprocessing mode:")
display(mode_counts)

print("\nOverall frozen baseline by preprocessing mode:")
display(overall_by_mode)

print("\nValidation/test frozen baseline by preprocessing mode:")
display(val_test_by_mode)

print("\nSource-FPS live-compatible baseline by dataset and role:")
display(source_fps_by_dataset_role)

print("\nPrimary fine-tuning targets to beat:")
display(source_fps_targets)

if len(baseline_predictions) != expected_prediction_rows:
    raise AssertionError(
        f"Expected {expected_prediction_rows} prediction rows, "
        f"found {len(baseline_predictions)}."
    )

if sorted(baseline_predictions["preprocessing_mode"].unique().tolist()) != sorted(expected_modes):
    raise AssertionError("Baseline prediction modes do not match the expected modes.")

if not (status_counts["preprocessing_status"] == "ok").all():
    raise AssertionError("At least one baseline preprocessing mode has failed rows.")

if not (mode_counts["rows"] == expected_eval_windows).all():
    raise AssertionError("Not every preprocessing mode has one row per evaluation window.")

print("\nPASS: Frozen baseline files match the expected live-compatible evaluation contract.")

Prediction shape check:
{
  "baseline_summary_rows": 168,
  "baseline_prediction_rows": 76208,
  "expected_prediction_rows": 76208,
  "expected_eval_windows": 19052,
  "unique_modes": [
    "stored_reference",
    "training_buffer_sim_10fps",
    "training_buffer_sim_15fps",
    "training_buffer_source_fps"
  ]
}

Status counts:


,preprocessing_mode,preprocessing_status,rows
0,stored_reference,ok,19052
1,training_buffer_sim_10fps,ok,19052
2,training_buffer_sim_15fps,ok,19052
3,training_buffer_source_fps,ok,19052



Rows per preprocessing mode:


,preprocessing_mode,rows
0,stored_reference,19052
1,training_buffer_sim_10fps,19052
2,training_buffer_sim_15fps,19052
3,training_buffer_source_fps,19052



Overall frozen baseline by preprocessing mode:


...


Source-FPS live-compatible baseline by dataset and role:


,dataset,window_role,rows,model_mae_mean,model_mae_median,model_mae_p90,spectral_mae_mean,spectral_mae_median,spectral_mae_p90,spectral_spread_mean,target_hr_mean,target_hr_range_mean
1,mcd_rppg,finetune_test,2094,4.88,3.15,10.97,10.29,6.15,22.61,9.37,79.85,12.41
4,ubfc_phys,finetune_test,658,8.86,5.67,20.29,13.94,10.27,32.17,18.14,77.42,78.73
7,ubfc_rppg,finetune_test,76,3.41,3.15,6.08,3.46,3.25,6.75,1.78,98.27,5.20
2,mcd_rppg,finetune_train,7446,4.69,3.16,10.37,10.60,5.36,24.75,11.43,79.06,14.29
5,ubfc_phys,finetune_train,4595,7.78,5.42,18.59,13.23,8.71,32.57,17.37,80.31,65.28
8,ubfc_rppg,finetune_train,462,4.65,3.14,9.92,4.93,3.23,9.65,5.02,98.45,4.48
3,mcd_rppg,finetune_val,1743,5.13,3.37,11.54,10.79,4.74,27.12,13.21,79.11,17.89
6,ubfc_phys,finetune_val,986,7.76,4.85,18.27,10.93,7.00,27.51,14.86,80.00,64.59
9,ubfc_rppg,finetune_val,110,4.65,2.44,12.38,4.54,3.04,12.50,3.14,98.28,5.75
0,ecg_fitness,ood_eval,882,14.04,7.68,38.15,27.30,22.73,61.63,29.20,109.20,11.72



Primary fine-tuning targets to beat:


,window_role,rows,model_mae_mean,model_mae_median,model_mae_p90,spectral_mae_mean,spectral_mae_median,spectral_mae_p90
0,finetune_test,2828,5.77,3.63,13.70,10.96,6.97,24.83
1,finetune_val,2839,6.03,3.85,14.07,10.59,5.31,26.95



PASS: Frozen baseline files match the expected live-compatible evaluation contract.


## HDF5 Schema And Single-Window Tensor Check

This section verifies that one manifest row can be resolved back to the correct HDF5 recording.

The notebook checks:

- the dataset file,
- the subject group,
- the recording group,
- the source frame rate,
- the stored POS, CHROM, and GREEN signals,
- the ROI RGB array,
- the window start/end frame indices,
- and the HR label stored in the manifest.

This is the bridge between table metadata and model input.

The expected model input shape is:

```text
(3, 240)

In [4]:



CHANNELS = ("pos", "chrom", "green")
H5_SIGNAL_NAMES = {
    "pos": "rppg_pos",
    "chrom": "rppg_chrom",
    "green": "rppg_green",
}
TARGET_FRAMES = 240


def zscore_signal(signal: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Return a per-window z-scored signal as float32."""
    signal = np.asarray(signal, dtype=np.float32)

    mean = float(np.mean(signal))
    std = float(np.std(signal))

    if not np.isfinite(std) or std < eps:
        raise ValueError("Cannot z-score a flat or invalid signal window.")

    return ((signal - mean) / (std + eps)).astype(np.float32)


def model_ready(signal: np.ndarray, target_frames: int = TARGET_FRAMES) -> np.ndarray:
    """Resample one signal window to target_frames and apply per-window z-score."""
    signal = np.asarray(signal, dtype=np.float32)

    if np.any(~np.isfinite(signal)):
        raise ValueError("Signal window contains NaN or infinite values.")

    if len(signal) != target_frames:
        signal = resample(signal, target_frames).astype(np.float32)

    return zscore_signal(signal)


def get_recording_group(row: pd.Series):
    """Open the HDF5 recording group referenced by one manifest row."""
    dataset_name = row["dataset"]
    subject_id = str(row["subject_id"])
    recording_id = str(row["recording_id"])

    h5_path = H5_PATHS[dataset_name]
    h5 = h5py.File(h5_path, "r")

    try:
        group = h5["subjects"][subject_id]["recordings"][recording_id]
    except KeyError:
        h5.close()
        raise KeyError(
            f"Could not find dataset={dataset_name}, "
            f"subject_id={subject_id}, recording_id={recording_id}"
        )

    return h5, group


def build_stored_reference_tensor(row: pd.Series) -> tuple[np.ndarray, dict]:
    """Build a stored-reference model tensor from POS, CHROM, and GREEN HDF5 signals."""
    start = int(row["start_frame"])
    end = int(row["end_frame"])

    h5, group = get_recording_group(row)

    try:
        channel_windows = []

        for channel in CHANNELS:
            dataset_name = H5_SIGNAL_NAMES[channel]
            raw_window = group[dataset_name][start:end].astype(np.float32)
            ready_window = model_ready(raw_window, target_frames=TARGET_FRAMES)
            channel_windows.append(ready_window)

        tensor = np.stack(channel_windows, axis=0).astype(np.float32)

        info = {
            "dataset": row["dataset"],
            "subject_id": str(row["subject_id"]),
            "recording_id": str(row["recording_id"]),
            "window_role": row["window_role"],
            "start_frame": start,
            "end_frame": end,
            "raw_window_frames": end - start,
            "tensor_shape": tensor.shape,
            "target_hr_mean": float(row["target_hr_mean"]),
            "target_hr_median": float(row["target_hr_median"]),
            "source_fps_manifest": float(row["source_fps"]),
            "fps_attr": float(group.attrs.get("fps", np.nan)),
            "available_datasets": sorted(list(group.keys())),
            "available_attrs": sorted(list(group.attrs.keys())),
        }

    finally:
        h5.close()

    return tensor, info


# Pick one real training window first.
example_row = (
    manifest
    .loc[manifest["window_role"] == "finetune_train"]
    .sort_values(["dataset", "subject_id", "recording_id", "start_frame"])
    .iloc[0]
)

stored_tensor, stored_info = build_stored_reference_tensor(example_row)

tensor_stats = pd.DataFrame(
    [
        {
            "channel": channel,
            "mean": float(stored_tensor[i].mean()),
            "std": float(stored_tensor[i].std()),
            "min": float(stored_tensor[i].min()),
            "max": float(stored_tensor[i].max()),
            "n_frames": stored_tensor[i].shape[0],
        }
        for i, channel in enumerate(CHANNELS)
    ]
)

print("Example window info:")
print(json.dumps(stored_info, indent=2))

print("\nStored-reference tensor stats:")
display(tensor_stats.round(4))

if stored_tensor.shape != (3, 240):
    raise AssertionError(f"Expected tensor shape (3, 240), found {stored_tensor.shape}")

if not np.isfinite(stored_tensor).all():
    raise AssertionError("Stored-reference tensor contains NaN or infinite values.")

print("\nPASS: One manifest row was resolved into a valid stored-reference (3, 240) tensor.")

Example window info:
{
  "dataset": "mcd_rppg",
  "subject_id": "1035",
  "recording_id": "before",
  "window_role": "finetune_train",
  "start_frame": 0,
  "end_frame": 239,
  "raw_window_frames": 239,
  "tensor_shape": [
    3,
    240
  ],
  "target_hr_mean": 71.26324462890625,
  "target_hr_median": 69.76819610595703,
  "source_fps_manifest": 29.9,
  "fps_attr": 29.9,
  "available_datasets": [
    "ensemble_weights",
    "hr_continuous",
    "reference_signal",
    "roi_rgb",
    "rppg_chrom",
    "rppg_ensemble",
    "rppg_green",
    "rppg_pos",
    "rr_intervals"
  ],
  "available_attrs": [
    "activity_id",
    "activity_name",
    "bm_age",
    "bm_age_tier",
    "bm_age_valid",
    "bm_arterial_stiffness",
    "bm_arterial_stiffness_tier",
    "bm_arterial_stiffness_valid",
    "bm_bmi",
    "bm_bmi_tier",
    "bm_bmi_valid",
    "bm_cholesterol",
    "bm_cholesterol_tier",
    "bm_cholesterol_valid",
    "bm_diastolic_bp",
    "bm_diastolic_bp_tier",
    "bm_diastolic_bp_val

,channel,mean,std,min,max,n_frames
0,pos,0.0,1.0,-2.0162,2.1675,240
1,chrom,-0.0,1.0,-2.7352,2.3568,240
2,green,-0.0,1.0,-2.4967,2.4681,240



PASS: One manifest row was resolved into a valid stored-reference (3, 240) tensor.


## Live-Compatible Source-FPS Tensor Check

This section rebuilds a model input from `roi_rgb` rather than from stored `rppg_pos`, `rppg_chrom`, and `rppg_green`.

The reconstructed path matches the frozen-baseline mode:

```text
training_buffer_source_fps

In [5]:
BUFFER_SEC = 12.0
BANDPASS_LOW_HZ = 0.7
BANDPASS_HIGH_HZ = 3.5
BANDPASS_ORDER = 4


def zscore_training_signal(signal: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Return a z-scored signal, matching the audit helper behavior for flat inputs."""
    signal = np.asarray(signal, dtype=np.float64)
    std = float(np.std(signal))

    if not np.isfinite(std) or std < eps:
        return np.zeros_like(signal, dtype=np.float32)

    return ((signal - float(np.mean(signal))) / std).astype(np.float32)


def bandpass_filter(
    signal: np.ndarray,
    fps: float,
    low_hz: float = BANDPASS_LOW_HZ,
    high_hz: float = BANDPASS_HIGH_HZ,
    order: int = BANDPASS_ORDER,
) -> np.ndarray:
    """Apply the training-style heart-rate bandpass filter to one signal."""
    signal = np.asarray(signal, dtype=np.float64)
    nyquist = 0.5 * float(fps)

    if high_hz >= nyquist:
        raise ValueError(
            f"high_hz={high_hz} must be below Nyquist frequency {nyquist:.3f}. "
            f"Increase sampling rate or lower the bandpass high cutoff."
        )

    b, a = butter(order, [low_hz / nyquist, high_hz / nyquist], btype="band")
    min_len = 3 * max(len(a), len(b))

    if len(signal) <= min_len:
        raise ValueError(
            f"Signal too short for filtfilt bandpass: len={len(signal)}, required>{min_len}"
        )

    return filtfilt(b, a, signal).astype(np.float32)


def training_green_from_roi_rgb(roi_rgb: np.ndarray, fps: float) -> np.ndarray:
    """Build the training-style GREEN signal from ROI RGB data."""
    green_by_roi = roi_rgb[:, :, 1].astype(np.float64)
    signal = np.mean(green_by_roi, axis=1)
    return zscore_training_signal(bandpass_filter(signal, fps))


def training_pos_from_roi_rgb(roi_rgb: np.ndarray, fps: float) -> np.ndarray:
    """Build the training-style POS signal by computing POS per ROI before averaging."""
    projection = np.array(
        [
            [0.0, 1.0, -1.0],
            [-2.0, 1.0, 1.0],
        ],
        dtype=np.float64,
    )

    roi_signals = []

    for roi_index in range(roi_rgb.shape[1]):
        rgb = roi_rgb[:, roi_index, :].astype(np.float64)
        normalized = rgb / (np.mean(rgb, axis=0, keepdims=True) + 1e-8)

        projected = projection @ normalized.T
        s0 = projected[0]
        s1 = projected[1]

        alpha = np.std(s0) / (np.std(s1) + 1e-8)
        roi_signals.append(s0 + alpha * s1)

    signal = np.mean(np.stack(roi_signals, axis=0), axis=0)
    return zscore_training_signal(bandpass_filter(signal, fps))


def training_chrom_from_roi_rgb(roi_rgb: np.ndarray, fps: float) -> np.ndarray:
    """Build the training-style CHROM signal by computing CHROM per ROI before averaging."""
    roi_signals = []

    for roi_index in range(roi_rgb.shape[1]):
        rgb = roi_rgb[:, roi_index, :].astype(np.float64)
        normalized = rgb / (np.mean(rgb, axis=0, keepdims=True) + 1e-8)

        r = normalized[:, 0]
        g = normalized[:, 1]
        b = normalized[:, 2]

        x = 3.0 * r - 2.0 * g
        y = 1.5 * r + g - 1.5 * b

        alpha = np.std(x) / (np.std(y) + 1e-8)
        roi_signals.append(x - alpha * y)

    signal = np.mean(np.stack(roi_signals, axis=0), axis=0)
    return zscore_training_signal(bandpass_filter(signal, fps))


def build_training_style_signals(roi_rgb: np.ndarray, fps: float) -> dict[str, np.ndarray]:
    """Compute POS, CHROM, and GREEN from ROI RGB using the training-style ROI pipeline."""
    return {
        "pos": training_pos_from_roi_rgb(roi_rgb, fps),
        "chrom": training_chrom_from_roi_rgb(roi_rgb, fps),
        "green": training_green_from_roi_rgb(roi_rgb, fps),
    }


def signal_corr(left: np.ndarray, right: np.ndarray) -> float:
    """Return Pearson correlation for two same-length signals."""
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if len(left) != len(right):
        raise ValueError(f"Length mismatch: {len(left)} vs {len(right)}")

    if np.std(left) < 1e-8 or np.std(right) < 1e-8:
        return float("nan")

    return float(np.corrcoef(left, right)[0, 1])


def build_source_fps_live_compatible_tensor(
    row: pd.Series,
    buffer_seconds: float = BUFFER_SEC,
) -> tuple[np.ndarray, dict[str, np.ndarray], dict]:
    """Build the source-FPS local-buffer model tensor for one manifest row."""
    start = int(row["start_frame"])
    end = int(row["end_frame"])
    source_fps = float(row["source_fps"])

    h5, group = get_recording_group(row)

    try:
        if "roi_rgb" not in group:
            raise KeyError("Recording does not contain roi_rgb.")

        roi_rgb_full = group["roi_rgb"][:].astype(np.float32)
        n_frames = int(roi_rgb_full.shape[0])

        if start < 0 or end > n_frames or start >= end:
            raise ValueError(
                f"Invalid manifest window start={start}, end={end}, n_frames={n_frames}"
            )

        window_frames = end - start

        # Match the full frozen-baseline script:
        # buffer_frames = int(source_fps * buffer_seconds), not round(...).
        buffer_frames = int(source_fps * buffer_seconds)
        buffer_start = max(0, end - buffer_frames)

        roi_rgb_buffer = roi_rgb_full[buffer_start:end]
        buffer_signals = build_training_style_signals(roi_rgb_buffer, source_fps)

        relative_start = max(0, len(roi_rgb_buffer) - window_frames)

        source_signals = {
            channel: model_ready(
                buffer_signals[channel][relative_start:],
                target_frames=TARGET_FRAMES,
            )
            for channel in CHANNELS
        }

        tensor = np.stack(
            [source_signals[channel] for channel in CHANNELS],
            axis=0,
        ).astype(np.float32)

        info = {
            "dataset": row["dataset"],
            "subject_id": str(row["subject_id"]),
            "recording_id": str(row["recording_id"]),
            "window_role": row["window_role"],
            "start_frame": start,
            "end_frame": end,
            "window_frames": window_frames,
            "source_fps": source_fps,
            "buffer_seconds_requested": buffer_seconds,
            "buffer_frames_requested": buffer_frames,
            "buffer_start_frame": int(buffer_start),
            "buffer_end_frame": int(end),
            "buffer_sample_count": int(len(roi_rgb_buffer)),
            "actual_buffer_seconds": float(len(roi_rgb_buffer) / source_fps),
            "relative_window_start_in_buffer": int(relative_start),
            "roi_rgb_full_shape": tuple(int(x) for x in roi_rgb_full.shape),
            "roi_rgb_buffer_shape": tuple(int(x) for x in roi_rgb_buffer.shape),
            "tensor_shape": tuple(int(x) for x in tensor.shape),
        }

    finally:
        h5.close()

    return tensor, source_signals, info


source_tensor, source_signals, source_info = build_source_fps_live_compatible_tensor(example_row)

source_tensor_stats = pd.DataFrame(
    [
        {
            "channel": channel,
            "mean": float(source_tensor[i].mean()),
            "std": float(source_tensor[i].std()),
            "min": float(source_tensor[i].min()),
            "max": float(source_tensor[i].max()),
            "n_frames": source_tensor[i].shape[0],
        }
        for i, channel in enumerate(CHANNELS)
    ]
)

stored_vs_source = pd.DataFrame(
    [
        {
            "channel": channel,
            "corr_vs_stored": signal_corr(stored_tensor[i], source_tensor[i]),
            "mae_vs_stored": float(np.mean(np.abs(stored_tensor[i] - source_tensor[i]))),
            "rmse_vs_stored": float(np.sqrt(np.mean((stored_tensor[i] - source_tensor[i]) ** 2))),
        }
        for i, channel in enumerate(CHANNELS)
    ]
)

print("Source-FPS live-compatible window info:")
print(json.dumps(source_info, indent=2))

print("\nSource-FPS tensor stats:")
display(source_tensor_stats.round(4))

print("\nStored-reference vs source-FPS tensor comparison:")
display(stored_vs_source.round(4))

if source_tensor.shape != (3, 240):
    raise AssertionError(f"Expected source tensor shape (3, 240), found {source_tensor.shape}")

if not np.isfinite(source_tensor).all():
    raise AssertionError("Source-FPS tensor contains NaN or infinite values.")

print("\nPASS: One manifest row was resolved into a valid source-FPS live-compatible (3, 240) tensor.")

Source-FPS live-compatible window info:
{
  "dataset": "mcd_rppg",
  "subject_id": "1035",
  "recording_id": "before",
  "window_role": "finetune_train",
  "start_frame": 0,
  "end_frame": 239,
  "window_frames": 239,
  "source_fps": 29.9,
  "buffer_seconds_requested": 12.0,
  "buffer_frames_requested": 358,
  "buffer_start_frame": 0,
  "buffer_end_frame": 239,
  "buffer_sample_count": 239,
  "actual_buffer_seconds": 7.993311036789298,
  "relative_window_start_in_buffer": 0,
  "roi_rgb_full_shape": [
    5352,
    3,
    3
  ],
  "roi_rgb_buffer_shape": [
    239,
    3,
    3
  ],
  "tensor_shape": [
    3,
    240
  ]
}

Source-FPS tensor stats:


,channel,mean,std,min,max,n_frames
0,pos,-0.0,1.0,-2.0435,2.2199,240
1,chrom,0.0,1.0,-2.4426,2.1215,240
2,green,-0.0,1.0,-2.4544,2.4238,240



Stored-reference vs source-FPS tensor comparison:


,channel,corr_vs_stored,mae_vs_stored,rmse_vs_stored
0,pos,0.8439,0.4136,0.5588
1,chrom,0.9398,0.2785,0.3469
2,green,0.9969,0.0406,0.0793



PASS: One manifest row was resolved into a valid source-FPS live-compatible (3, 240) tensor.


## Frozen PhysFormer Checkpoint Sanity Check

This section loads the current multichannel CRVSE PhysFormer checkpoint and runs inference on the two tensors built above:

- stored-reference tensor,
- source-FPS live-compatible tensor.

The purpose is not to train yet.

The purpose is to verify the full path:

```text
manifest row -> HDF5 recording -> model tensor -> PhysFormer checkpoint -> HR prediction

In [6]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformer time tokens."""

    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.size(1) > self.pe.size(1):
            raise ValueError(
                f"Input sequence length {x.size(1)} exceeds positional encoding "
                f"max length {self.pe.size(1)}."
            )

        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


class TransformerEncoderLayerCustom(nn.Module):
    """Custom pre-norm transformer encoder layer used by the CRVSE checkpoint."""

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}.")

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _attention(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, time_steps, -1)

        return self.out_proj(out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self._attention(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class CRVSEPhysFormer(nn.Module):
    """CNN + FFT + Transformer model for rPPG heart-rate estimation."""

    def __init__(
        self,
        in_channels: int = 3,
        cnn_channels: int = 16,
        freq_channels: int = 64,
        n_heads: int = 4,
        n_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.11331939348791525,
        hr_min: float = 40.0,
        hr_max: float = 180.0,
        target_frames: int = 240,
        max_positional_length: int = 300,
    ) -> None:
        super().__init__()

        self.in_channels = in_channels
        self.hr_min = hr_min
        self.hr_max = hr_max
        self.target_frames = target_frames
        self.d_model = cnn_channels + freq_channels

        if self.d_model % n_heads != 0:
            raise ValueError(f"d_model={self.d_model} must be divisible by n_heads={n_heads}.")

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, cnn_channels // 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels // 2, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        n_fft_bins = target_frames // 2 + 1

        self.freq_proj = nn.Sequential(
            nn.Linear(n_fft_bins, freq_channels * 4),
            nn.ReLU(),
            nn.Linear(freq_channels * 4, freq_channels),
        )

        self.pos_enc = PositionalEncoding(
            d_model=self.d_model,
            max_len=max_positional_length,
            dropout=dropout,
        )

        self.transformer_layers = nn.ModuleList(
            [
                TransformerEncoderLayerCustom(
                    d_model=self.d_model,
                    n_heads=n_heads,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                )
                for _ in range(n_layers)
            ]
        )

        self.head = nn.Sequential(
            nn.Linear(self.d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected x shape (batch, channels, time), got {tuple(x.shape)}.")

        batch_size, channels, time_steps = x.shape

        if channels != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} channels, got {channels}.")

        if time_steps != self.target_frames:
            raise ValueError(f"Expected {self.target_frames} frames, got {time_steps}.")

        time_feat = self.encoder(x)
        time_feat = time_feat.permute(0, 2, 1)

        freq = torch.fft.rfft(x, norm="ortho")
        freq_mag = freq.abs().mean(dim=1)
        freq_feat = self.freq_proj(freq_mag)
        freq_feat = freq_feat.unsqueeze(1).expand(-1, time_steps, -1)

        combined = torch.cat([time_feat, freq_feat], dim=-1)
        combined = self.pos_enc(combined)

        for layer in self.transformer_layers:
            combined = layer(combined)

        out = combined.mean(dim=1)
        out = self.head(out).squeeze(-1)

        if not self.training:
            out = out.clamp(self.hr_min, self.hr_max)

        return out


def count_parameters(model: nn.Module) -> int:
    """Count trainable model parameters."""
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def load_crvse_checkpoint(path: Path, device: torch.device) -> dict:
    """Load the CRVSE checkpoint dictionary."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = load_crvse_checkpoint(CHECKPOINT_PATH, DEVICE)
state_dict = checkpoint["model_state"]
best_params = checkpoint["best_params"]

dim_feedforward = int(state_dict["transformer_layers.0.ff.0.weight"].shape[0])

model = CRVSEPhysFormer(
    in_channels=int(checkpoint["in_channels"]),
    cnn_channels=int(best_params["cnn_channels"]),
    freq_channels=int(best_params["freq_channels"]),
    n_heads=int(best_params["n_heads"]),
    n_layers=int(best_params["n_layers"]),
    dim_feedforward=dim_feedforward,
    dropout=float(best_params["dropout"]),
    hr_min=40.0,
    hr_max=180.0,
    target_frames=TARGET_FRAMES,
    max_positional_length=300,
).to(DEVICE)

model.load_state_dict(state_dict, strict=True)
model.eval()


def predict_hr_from_numpy_tensor(tensor: np.ndarray, model: nn.Module) -> float:
    """Run model inference for one numpy tensor with shape (3, 240)."""
    x = torch.tensor(tensor, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    with torch.inference_mode():
        prediction = model(x)

    return float(prediction.detach().cpu().numpy()[0])


stored_reference_hr = predict_hr_from_numpy_tensor(stored_tensor, model)
source_fps_hr = predict_hr_from_numpy_tensor(source_tensor, model)

same_window_mask = (
    (baseline_predictions["dataset"].astype(str) == str(example_row["dataset"]))
    & (baseline_predictions["subject_id"].astype(str) == str(example_row["subject_id"]))
    & (baseline_predictions["recording_id"].astype(str) == str(example_row["recording_id"]))
    & (baseline_predictions["start_frame"].astype(int) == int(example_row["start_frame"]))
    & (baseline_predictions["end_frame"].astype(int) == int(example_row["end_frame"]))
)

baseline_same_window = baseline_predictions.loc[
    same_window_mask,
    [
        "preprocessing_mode",
        "target_hr_mean",
        "model_hr",
        "model_abs_error",
        "spectral_consensus_hr",
        "spectral_abs_error",
    ],
].copy()

baseline_same_window["model_hr"] = pd.to_numeric(
    baseline_same_window["model_hr"],
    errors="coerce",
)

notebook_prediction_rows = pd.DataFrame(
    [
        {
            "preprocessing_mode": "stored_reference",
            "notebook_model_hr": stored_reference_hr,
        },
        {
            "preprocessing_mode": "training_buffer_source_fps",
            "notebook_model_hr": source_fps_hr,
        },
    ]
)

prediction_check = notebook_prediction_rows.merge(
    baseline_same_window,
    on="preprocessing_mode",
    how="left",
)

prediction_check["abs_diff_vs_baseline"] = (prediction_check["notebook_model_hr"] - prediction_check["model_hr"]).abs()

checkpoint_report = {
    "checkpoint_input_mode": checkpoint.get("input_mode"),
    "checkpoint_in_channels": checkpoint.get("in_channels"),
    "checkpoint_best_val_mae": checkpoint.get("best_val_mae"),
    "checkpoint_best_n_epochs": checkpoint.get("best_n_epochs"),
    "best_params": best_params,
    "dim_feedforward_from_state_dict": dim_feedforward,
    "trainable_parameters": count_parameters(model),
    "device": str(DEVICE),
}

print("Checkpoint report:")
print(json.dumps(checkpoint_report, indent=2))

print("\nBaseline rows for the example window:")
display(baseline_same_window.round(4))

print("\nNotebook prediction check:")
display(prediction_check.round(4))

max_allowed_diff_bpm = 0.50
if prediction_check["model_hr"].isna().any():
    raise AssertionError("Could not find matching baseline rows for the example window.")

if (prediction_check["abs_diff_vs_baseline"] > max_allowed_diff_bpm).any():
    raise AssertionError(
        "Notebook predictions differ from the baseline CSV by more than "
        f"{max_allowed_diff_bpm} BPM."
    )

print("\nPASS: Frozen PhysFormer inference matches the baseline CSV for this example window.")

Checkpoint report:
{
  "checkpoint_input_mode": "multichannel",
  "checkpoint_in_channels": 3,
  "checkpoint_best_val_mae": 6.900240182876587,
  "checkpoint_best_n_epochs": 50,
  "best_params": {
    "lr": 0.0006523895417699133,
    "weight_decay": 0.0008816216939148644,
    "dropout": 0.11331939348791525,
    "huber_delta": 6.544152619447903,
    "cnn_channels": 16,
    "freq_channels": 64,
    "n_heads": 4,
    "n_layers": 4,
    "ff_mult": 4
  },
  "dim_feedforward_from_state_dict": 256,
  "trainable_parameters": 322145,
  "device": "cuda"
}

Baseline rows for the example window:


,preprocessing_mode,target_hr_mean,model_hr,model_abs_error,spectral_consensus_hr,spectral_abs_error
176,stored_reference,71.2632,70.9094,0.3539,67.5,3.7632
177,training_buffer_source_fps,71.2632,73.8221,2.5589,67.5,3.7632
178,training_buffer_sim_15fps,71.2632,75.6790,4.4158,67.5,3.7632
179,training_buffer_sim_10fps,71.2632,77.2375,5.9742,67.5,3.7632



Notebook prediction check:


,preprocessing_mode,notebook_model_hr,target_hr_mean,model_hr,model_abs_error,spectral_consensus_hr,spectral_abs_error,abs_diff_vs_baseline
0,stored_reference,70.9094,71.2632,70.9094,0.3539,67.5,3.7632,0.0
1,training_buffer_source_fps,73.8221,71.2632,73.8221,2.5589,67.5,3.7632,0.0



PASS: Frozen PhysFormer inference matches the baseline CSV for this example window.


## Manifest-Backed Dataset And One-Batch Check

This section builds the first PyTorch `Dataset` for live-compatible fine-tuning.

The dataset uses the manifest as the source of truth. It does not rebuild train/validation/test splits.

Each item returns:

```text
x: tensor with shape (3, 240)
y: target HR in BPM
metadata: dataset, subject, recording, role, frame range

In [7]:
class LiveCompatibleManifestDataset(Dataset):
    """PyTorch Dataset that builds source-FPS live-compatible tensors from manifest rows."""

    def __init__(
        self,
        manifest_df: pd.DataFrame,
        role: str,
        max_rows: int | None = None,
    ) -> None:
        self.role = role

        selected = manifest_df.loc[
            manifest_df["window_role"] == role
        ].copy()

        if role == "finetune_train":
            selected = selected.loc[selected["include_in_finetune_bool"]]
        else:
            selected = selected.loc[selected["include_in_eval_bool"]]

        selected = selected.sort_values(
            ["dataset", "subject_id", "recording_id", "start_frame"]
        ).reset_index(drop=True)

        if max_rows is not None:
            selected = selected.head(max_rows).reset_index(drop=True)

        if len(selected) == 0:
            raise ValueError(f"No rows found for role={role!r}")

        self.rows = selected

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, dict]:
        row = self.rows.iloc[index]

        tensor_np, _, info = build_source_fps_live_compatible_tensor(row)
        target_hr = float(row["target_hr_mean"])

        if not np.isfinite(target_hr):
            raise ValueError(f"Non-finite target HR at dataset index {index}")

        x = torch.tensor(tensor_np, dtype=torch.float32)
        y = torch.tensor(target_hr, dtype=torch.float32)

        metadata = {
            "dataset": str(row["dataset"]),
            "subject_id": str(row["subject_id"]),
            "recording_id": str(row["recording_id"]),
            "window_role": str(row["window_role"]),
            "start_frame": int(row["start_frame"]),
            "end_frame": int(row["end_frame"]),
            "target_hr_mean": target_hr,
            "label_stability_bucket": str(row["label_stability_bucket"]),
            "source_fps": float(row["source_fps"]),
            "actual_buffer_seconds": float(info["actual_buffer_seconds"]),
            "buffer_sample_count": int(info["buffer_sample_count"]),
        }

        return x, y, metadata


def collate_with_metadata(batch):
    """Collate tensors and labels while keeping metadata as a list of dictionaries."""
    xs, ys, metadata = zip(*batch)

    return (
        torch.stack(xs, dim=0),
        torch.stack(ys, dim=0),
        list(metadata),
    )


# Small subsets first. This checks correctness without scanning the whole dataset.
train_check_ds = LiveCompatibleManifestDataset(
    manifest_df=manifest,
    role="finetune_train",
    max_rows=32,
)

val_check_ds = LiveCompatibleManifestDataset(
    manifest_df=manifest,
    role="finetune_val",
    max_rows=32,
)

train_check_loader = DataLoader(
    train_check_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_with_metadata,
)

val_check_loader = DataLoader(
    val_check_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_with_metadata,
)

train_x, train_y, train_meta = next(iter(train_check_loader))
val_x, val_y, val_meta = next(iter(val_check_loader))

batch_report = {
    "train_subset_rows": len(train_check_ds),
    "val_subset_rows": len(val_check_ds),
    "train_batch_x_shape": tuple(train_x.shape),
    "train_batch_y_shape": tuple(train_y.shape),
    "val_batch_x_shape": tuple(val_x.shape),
    "val_batch_y_shape": tuple(val_y.shape),
    "train_x_finite": bool(torch.isfinite(train_x).all().item()),
    "train_y_finite": bool(torch.isfinite(train_y).all().item()),
    "val_x_finite": bool(torch.isfinite(val_x).all().item()),
    "val_y_finite": bool(torch.isfinite(val_y).all().item()),
    "train_y_min": float(train_y.min().item()),
    "train_y_max": float(train_y.max().item()),
    "val_y_min": float(val_y.min().item()),
    "val_y_max": float(val_y.max().item()),
}

train_channel_stats = pd.DataFrame(
    [
        {
            "channel": CHANNELS[channel_index],
            "batch_mean": float(train_x[:, channel_index, :].mean().item()),
            "batch_std": float(train_x[:, channel_index, :].std().item()),
            "batch_min": float(train_x[:, channel_index, :].min().item()),
            "batch_max": float(train_x[:, channel_index, :].max().item()),
        }
        for channel_index in range(train_x.shape[1])
    ]
)

train_metadata_preview = pd.DataFrame(train_meta)

print("Batch report:")
print(json.dumps(batch_report, indent=2))

print("\nTrain batch channel stats:")
display(train_channel_stats.round(4))

print("\nTrain metadata preview:")
display(train_metadata_preview)

if train_x.shape != (8, 3, 240):
    raise AssertionError(f"Expected train batch shape (8, 3, 240), found {tuple(train_x.shape)}")

if val_x.shape != (8, 3, 240):
    raise AssertionError(f"Expected val batch shape (8, 3, 240), found {tuple(val_x.shape)}")

if not torch.isfinite(train_x).all() or not torch.isfinite(train_y).all():
    raise AssertionError("Train batch contains non-finite values.")

if not torch.isfinite(val_x).all() or not torch.isfinite(val_y).all():
    raise AssertionError("Validation batch contains non-finite values.")

print("\nPASS: Manifest-backed live-compatible Dataset returns valid train/val batches.")

Batch report:
{
  "train_subset_rows": 32,
  "val_subset_rows": 32,
  "train_batch_x_shape": [
    8,
    3,
    240
  ],
  "train_batch_y_shape": [
    8
  ],
  "val_batch_x_shape": [
    8,
    3,
    240
  ],
  "val_batch_y_shape": [
    8
  ],
  "train_x_finite": true,
  "train_y_finite": true,
  "val_x_finite": true,
  "val_y_finite": true,
  "train_y_min": 65.60918426513672,
  "train_y_max": 71.26324462890625,
  "val_y_min": 91.69681549072266,
  "val_y_max": 97.57855224609375
}

Train batch channel stats:


,channel,batch_mean,batch_std,batch_min,batch_max
0,pos,-0.0,1.0003,-3.2048,2.8079
1,chrom,0.0,1.0003,-2.9748,3.0915
2,green,-0.0,1.0003,-2.4881,2.5915



Train metadata preview:


,dataset,subject_id,recording_id,window_role,start_frame,end_frame,target_hr_mean,label_stability_bucket,source_fps,actual_buffer_seconds,buffer_sample_count
0,mcd_rppg,1035,before,finetune_train,0,239,71.263245,moderate_10_25_bpm,29.9,7.993311,239
1,mcd_rppg,1035,before,finetune_train,119,358,67.810760,stable_lt_10_bpm,29.9,11.973244,358
2,mcd_rppg,1035,before,finetune_train,238,477,66.663406,moderate_10_25_bpm,29.9,11.973244,358
3,mcd_rppg,1035,before,finetune_train,357,596,65.675186,moderate_10_25_bpm,29.9,11.973244,358
4,mcd_rppg,1035,before,finetune_train,476,715,65.609184,stable_lt_10_bpm,29.9,11.973244,358
5,mcd_rppg,1035,before,finetune_train,595,834,66.315422,stable_lt_10_bpm,29.9,11.973244,358
6,mcd_rppg,1035,before,finetune_train,714,953,66.319603,stable_lt_10_bpm,29.9,11.973244,358
7,mcd_rppg,1035,before,finetune_train,833,1072,67.550240,moderate_10_25_bpm,29.9,11.973244,358



PASS: Manifest-backed live-compatible Dataset returns valid train/val batches.


## Tiny Overfit Test

This section performs a small training sanity check on 32 live-compatible training windows.

The purpose is not to produce a useful model.

The purpose is to verify that:

- the checkpoint can be copied into a trainable model,
- the loss function receives valid predictions and labels,
- gradients flow through the model,
- the optimizer updates weights,
- and training loss decreases on a tiny subset.

A successful tiny overfit test does not prove generalization. It only proves that the training path is mechanically valid.

In [8]:
def materialize_dataset(dataset: Dataset) -> tuple[torch.Tensor, torch.Tensor, list[dict]]:
    """Load a small Dataset fully into memory for controlled debugging."""
    xs = []
    ys = []
    metadata = []

    for index in range(len(dataset)):
        x, y, meta = dataset[index]
        xs.append(x)
        ys.append(y)
        metadata.append(meta)

    return torch.stack(xs, dim=0), torch.stack(ys, dim=0), metadata


tiny_train_x, tiny_train_y, tiny_train_meta = materialize_dataset(train_check_ds)

tiny_train_x = tiny_train_x.to(DEVICE)
tiny_train_y = tiny_train_y.to(DEVICE)

overfit_model = copy.deepcopy(model).to(DEVICE)
overfit_model.train()

loss_fn = nn.HuberLoss(delta=float(best_params["huber_delta"]))
optimizer = AdamW(
    overfit_model.parameters(),
    lr=1e-4,
    weight_decay=0.0,
)

overfit_history = []

for epoch in range(1, 51):
    optimizer.zero_grad(set_to_none=True)

    predictions = overfit_model(tiny_train_x)
    loss = loss_fn(predictions, tiny_train_y)

    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(overfit_model.parameters(), max_norm=1.0)

    optimizer.step()

    with torch.no_grad():
        mae = torch.mean(torch.abs(predictions - tiny_train_y))

    overfit_history.append(
        {
            "epoch": epoch,
            "loss": float(loss.detach().cpu().item()),
            "mae": float(mae.detach().cpu().item()),
            "grad_norm": float(grad_norm.detach().cpu().item()),
        }
    )

overfit_history_df = pd.DataFrame(overfit_history)

overfit_report = {
    "tiny_train_rows": int(tiny_train_x.shape[0]),
    "x_shape": tuple(tiny_train_x.shape),
    "y_shape": tuple(tiny_train_y.shape),
    "initial_loss": float(overfit_history_df["loss"].iloc[0]),
    "final_loss": float(overfit_history_df["loss"].iloc[-1]),
    "initial_mae": float(overfit_history_df["mae"].iloc[0]),
    "final_mae": float(overfit_history_df["mae"].iloc[-1]),
    "min_mae": float(overfit_history_df["mae"].min()),
}

print("Tiny overfit report:")
print(json.dumps(overfit_report, indent=2))

display(overfit_history_df.head(10).round(4))
display(overfit_history_df.tail(10).round(4))

if not np.isfinite(overfit_history_df["loss"]).all():
    raise AssertionError("Tiny overfit loss contains non-finite values.")

if overfit_history_df["loss"].iloc[-1] >= overfit_history_df["loss"].iloc[0]:
    raise AssertionError("Tiny overfit loss did not decrease.")

print("\nPASS: Tiny overfit test reduced loss on a small live-compatible subset.")

Tiny overfit report:
{
  "tiny_train_rows": 32,
  "x_shape": [
    32,
    3,
    240
  ],
  "y_shape": [
    32
  ],
  "initial_loss": 27.149066925048828,
  "final_loss": 17.604598999023438,
  "initial_mae": 6.746175765991211,
  "final_mae": 5.200354099273682,
  "min_mae": 3.441432476043701
}


,epoch,loss,mae,grad_norm
0,1,27.1491,6.7462,610.8645
1,2,25.4867,6.6162,514.9306
2,3,33.2566,7.9040,577.6382
3,4,36.6758,8.3663,351.2123
4,5,22.1806,5.8604,197.3157
5,6,24.8950,6.4764,303.9511
6,7,18.1694,5.1926,213.0518
7,8,25.8758,6.5946,290.2277
8,9,19.3361,5.3690,375.8060
9,10,16.3813,4.7342,260.1334


,epoch,loss,mae,grad_norm
40,41,14.6526,4.4637,165.6450
41,42,20.6912,5.8102,185.3721
42,43,18.9022,5.5486,461.4333
43,44,18.8402,5.5361,93.1334
44,45,18.8388,5.2985,112.8930
45,46,19.8484,5.4999,89.3380
46,47,17.1260,5.0132,296.0016
47,48,23.1690,6.0273,84.7214
48,49,13.0672,3.9783,232.7692
49,50,17.6046,5.2004,405.7021



PASS: Tiny overfit test reduced loss on a small live-compatible subset.


##  Tensor Materialization Pilot

This section tests whether live-compatible source-FPS tensors can be precomputed safely.

The fine-tuning dataset currently rebuilds each tensor from HDF5 `roi_rgb` every time `__getitem__` is called. That is correct, but inefficient for real training.

This pilot materializes a small, role-balanced subset first.

The goals are:

- estimate preprocessing speed,
- confirm that all selected rows produce `(3, 240)` tensors,
- check for NaN or infinite values,
- preserve row metadata for later error analysis,
- and decide whether full materialization is safe.

This cell does not train the model.

In [9]:
def select_materialization_pilot_rows(
    manifest_df: pd.DataFrame,
    rows_per_role: int = 128,
) -> pd.DataFrame:
    """Select a small role-balanced subset for tensor materialization testing."""
    roles = [
        "finetune_train",
        "finetune_val",
        "finetune_test",
        "ood_eval",
    ]

    selected_parts = []

    for role in roles:
        role_rows = manifest_df.loc[
            (manifest_df["window_role"] == role)
            & manifest_df["include_in_eval_bool"]
        ].copy()

        role_rows = role_rows.sort_values(
            ["dataset", "subject_id", "recording_id", "start_frame"]
        )

        if len(role_rows) > rows_per_role:
            role_rows = role_rows.sample(
                n=rows_per_role,
                random_state=SEED,
            ).sort_values(
                ["dataset", "subject_id", "recording_id", "start_frame"]
            )

        selected_parts.append(role_rows)

    return pd.concat(selected_parts, axis=0).reset_index(drop=True)


def materialize_rows_to_arrays(rows: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Materialize manifest rows into source-FPS tensors, labels, and metadata."""
    xs = []
    ys = []
    metadata_rows = []
    failures = []

    start_time = time.perf_counter()

    for index, row in rows.iterrows():
        try:
            tensor_np, _, info = build_source_fps_live_compatible_tensor(row)

            if tensor_np.shape != (3, 240):
                raise ValueError(f"Unexpected tensor shape: {tensor_np.shape}")

            if not np.isfinite(tensor_np).all():
                raise ValueError("Tensor contains NaN or infinite values.")

            target_hr = float(row["target_hr_mean"])
            if not np.isfinite(target_hr):
                raise ValueError("Target HR is not finite.")

            xs.append(tensor_np.astype(np.float32))
            ys.append(target_hr)

            metadata_rows.append(
                {
                    "row_index": int(index),
                    "dataset": str(row["dataset"]),
                    "subject_id": str(row["subject_id"]),
                    "recording_id": str(row["recording_id"]),
                    "window_role": str(row["window_role"]),
                    "start_frame": int(row["start_frame"]),
                    "end_frame": int(row["end_frame"]),
                    "target_hr_mean": target_hr,
                    "label_stability_bucket": str(row["label_stability_bucket"]),
                    "source_fps": float(row["source_fps"]),
                    "actual_buffer_seconds": float(info["actual_buffer_seconds"]),
                    "buffer_sample_count": int(info["buffer_sample_count"]),
                }
            )

        except Exception as exc:
            failures.append(
                {
                    "row_index": int(index),
                    "dataset": str(row.get("dataset")),
                    "subject_id": str(row.get("subject_id")),
                    "recording_id": str(row.get("recording_id")),
                    "window_role": str(row.get("window_role")),
                    "start_frame": row.get("start_frame"),
                    "end_frame": row.get("end_frame"),
                    "error": repr(exc),
                }
            )

    elapsed_seconds = time.perf_counter() - start_time

    if failures:
        failure_df = pd.DataFrame(failures)
        display(failure_df.head(20))
        raise RuntimeError(f"Materialization failed for {len(failures)} rows.")

    x_array = np.stack(xs, axis=0).astype(np.float32)
    y_array = np.asarray(ys, dtype=np.float32)
    metadata = pd.DataFrame(metadata_rows)

    metadata.attrs["elapsed_seconds"] = elapsed_seconds
    metadata.attrs["seconds_per_row"] = elapsed_seconds / max(len(metadata), 1)

    return x_array, y_array, metadata


pilot_rows = select_materialization_pilot_rows(
    manifest_df=manifest,
    rows_per_role=128,
)

pilot_x, pilot_y, pilot_metadata = materialize_rows_to_arrays(pilot_rows)

pilot_role_counts = (
    pilot_metadata
    .groupby(["dataset", "window_role"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["window_role", "dataset"])
)

pilot_channel_stats = pd.DataFrame(
    [
        {
            "channel": CHANNELS[channel_index],
            "mean": float(pilot_x[:, channel_index, :].mean()),
            "std": float(pilot_x[:, channel_index, :].std()),
            "min": float(pilot_x[:, channel_index, :].min()),
            "max": float(pilot_x[:, channel_index, :].max()),
        }
        for channel_index in range(pilot_x.shape[1])
    ]
)

seconds_per_row = float(pilot_metadata.attrs["seconds_per_row"])
expected_full_eval_rows = int(manifest["include_in_eval_bool"].sum())
estimated_full_minutes = seconds_per_row * expected_full_eval_rows / 60.0

pilot_report = {
    "pilot_rows": int(len(pilot_metadata)),
    "x_shape": tuple(pilot_x.shape),
    "y_shape": tuple(pilot_y.shape),
    "x_dtype": str(pilot_x.dtype),
    "y_dtype": str(pilot_y.dtype),
    "x_finite": bool(np.isfinite(pilot_x).all()),
    "y_finite": bool(np.isfinite(pilot_y).all()),
    "elapsed_seconds": float(pilot_metadata.attrs["elapsed_seconds"]),
    "seconds_per_row": seconds_per_row,
    "estimated_full_eval_minutes": estimated_full_minutes,
    "target_hr_min": float(pilot_y.min()),
    "target_hr_max": float(pilot_y.max()),
}

print("Pilot materialization report:")
print(json.dumps(pilot_report, indent=2))

print("\nPilot rows by dataset and role:")
display(pilot_role_counts)

print("\nPilot channel stats:")
display(pilot_channel_stats.round(4))

print("\nPilot metadata preview:")
display(pilot_metadata.head(10))

if pilot_x.shape[1:] != (3, 240):
    raise AssertionError(f"Expected tensor tail shape (3, 240), found {pilot_x.shape[1:]}")

if not np.isfinite(pilot_x).all() or not np.isfinite(pilot_y).all():
    raise AssertionError("Pilot materialization produced non-finite values.")

print("\nPASS: Tensor materialization pilot produced valid live-compatible tensors.")

Pilot materialization report:
{
  "pilot_rows": 512,
  "x_shape": [
    512,
    3,
    240
  ],
  "y_shape": [
    512
  ],
  "x_dtype": "float32",
  "y_dtype": "float32",
  "x_finite": true,
  "y_finite": true,
  "elapsed_seconds": 16.79998472700001,
  "seconds_per_row": 0.03281247016992189,
  "estimated_full_eval_minutes": 10.419053027955865,
  "target_hr_min": 41.43177795410156,
  "target_hr_max": 155.56369018554688
}

Pilot rows by dataset and role:


,dataset,window_role,rows
1,mcd_rppg,finetune_test,95
4,ubfc_phys,finetune_test,27
7,ubfc_rppg,finetune_test,6
2,mcd_rppg,finetune_train,75
5,ubfc_phys,finetune_train,49
8,ubfc_rppg,finetune_train,4
3,mcd_rppg,finetune_val,86
6,ubfc_phys,finetune_val,38
9,ubfc_rppg,finetune_val,4
0,ecg_fitness,ood_eval,128



Pilot channel stats:


,channel,mean,std,min,max
0,pos,0.0,1.0,-5.3537,5.6896
1,chrom,-0.0,1.0,-6.1681,6.6119
2,green,0.0,1.0,-7.4146,6.9539



Pilot metadata preview:


,row_index,dataset,subject_id,recording_id,window_role,start_frame,end_frame,target_hr_mean,label_stability_bucket,source_fps,actual_buffer_seconds,buffer_sample_count
0,0,mcd_rppg,1217,before,finetune_train,3570,3809,63.047878,stable_lt_10_bpm,29.9,11.973244,358
1,1,mcd_rppg,1305,before,finetune_train,3648,3840,108.142174,stable_lt_10_bpm,24.0,12.000000,288
2,2,mcd_rppg,1513,before,finetune_train,4760,4999,91.443825,unstable_25_50_bpm,29.9,11.973244,358
3,3,mcd_rppg,1541,after,finetune_train,1904,2143,75.901558,stable_lt_10_bpm,29.9,11.973244,358
4,4,mcd_rppg,2149,after,finetune_train,0,192,72.772392,unstable_25_50_bpm,24.0,8.000000,192
5,5,mcd_rppg,2149,after,finetune_train,1056,1248,63.430862,moderate_10_25_bpm,24.0,12.000000,288
6,6,mcd_rppg,2161,before,finetune_train,3456,3648,94.911095,moderate_10_25_bpm,24.0,12.000000,288
7,7,mcd_rppg,2183,after,finetune_train,2618,2857,74.013916,stable_lt_10_bpm,29.9,11.973244,358
8,8,mcd_rppg,2247,after,finetune_train,2592,2784,67.236122,stable_lt_10_bpm,24.0,12.000000,288
9,9,mcd_rppg,2414,after,finetune_train,3264,3456,82.832634,moderate_10_25_bpm,24.0,12.000000,288



PASS: Tensor materialization pilot produced valid live-compatible tensors.


## Full Tensor Materialization

This section materializes all live-compatible source-FPS tensors needed for fine-tuning and evaluation.

The notebook writes one tensor file per role:

```text
finetune_train
finetune_val
finetune_test
ood_eval

In [10]:
MATERIALIZED_DIR = KAGGLE_WORKING / "crvse_live_sourcefps_materialized"
MATERIALIZED_DIR.mkdir(parents=True, exist_ok=True)


def rows_for_role(manifest_df: pd.DataFrame, role: str) -> pd.DataFrame:
    """Return manifest rows for one training or evaluation role."""
    rows = manifest_df.loc[manifest_df["window_role"] == role].copy()

    if role == "finetune_train":
        rows = rows.loc[rows["include_in_finetune_bool"]]
    else:
        rows = rows.loc[rows["include_in_eval_bool"]]

    return rows.sort_values(
        ["dataset", "subject_id", "recording_id", "start_frame"]
    ).reset_index(drop=True)


def save_materialized_role(
    manifest_df: pd.DataFrame,
    role: str,
    output_dir: Path,
) -> dict:
    """Materialize and save tensors, labels, and metadata for one manifest role."""
    rows = rows_for_role(manifest_df, role)

    print(f"\nMaterializing {role}: {len(rows)} rows")
    start_time = time.perf_counter()

    x_array, y_array, metadata = materialize_rows_to_arrays(rows)

    tensor_path = output_dir / f"{role}_sourcefps_tensors.npz"
    metadata_path = output_dir / f"{role}_sourcefps_metadata.csv"

    np.savez(
        tensor_path,
        x=x_array,
        y=y_array,
    )
    metadata.to_csv(metadata_path, index=False)

    elapsed_seconds = time.perf_counter() - start_time

    report = {
        "role": role,
        "rows": int(len(metadata)),
        "x_shape": tuple(x_array.shape),
        "y_shape": tuple(y_array.shape),
        "x_dtype": str(x_array.dtype),
        "y_dtype": str(y_array.dtype),
        "target_hr_min": float(y_array.min()),
        "target_hr_max": float(y_array.max()),
        "elapsed_seconds": float(elapsed_seconds),
        "seconds_per_row": float(elapsed_seconds / max(len(metadata), 1)),
        "tensor_path": str(tensor_path),
        "metadata_path": str(metadata_path),
        "tensor_file_mb": float(tensor_path.stat().st_size / 1024 / 1024),
        "metadata_file_mb": float(metadata_path.stat().st_size / 1024 / 1024),
    }

    print(json.dumps(report, indent=2))
    return report


roles_to_materialize = [
    "finetune_train",
    "finetune_val",
    "finetune_test",
    "ood_eval",
]

materialization_reports = []

for role in roles_to_materialize:
    report = save_materialized_role(
        manifest_df=manifest,
        role=role,
        output_dir=MATERIALIZED_DIR,
    )
    materialization_reports.append(report)

materialization_report_df = pd.DataFrame(materialization_reports)

print("\nFull materialization report:")
display(materialization_report_df)

total_materialized_rows = int(materialization_report_df["rows"].sum())
expected_rows = int(manifest.loc[manifest["include_in_eval_bool"]].shape[0])

if total_materialized_rows != expected_rows:
    raise AssertionError(
        f"Expected {expected_rows} materialized rows, found {total_materialized_rows}."
    )

print("\nPASS: Full live-compatible source-FPS tensors were materialized successfully.")


Materializing finetune_train: 12503 rows
{
  "role": "finetune_train",
  "rows": 12503,
  "x_shape": [
    12503,
    3,
    240
  ],
  "y_shape": [
    12503
  ],
  "x_dtype": "float32",
  "y_dtype": "float32",
  "target_hr_min": 40.056312561035156,
  "target_hr_max": 127.0,
  "elapsed_seconds": 336.574241041,
  "seconds_per_row": 0.02691947860841398,
  "tensor_path": "/kaggle/working/crvse_live_sourcefps_materialized/finetune_train_sourcefps_tensors.npz",
  "metadata_path": "/kaggle/working/crvse_live_sourcefps_materialized/finetune_train_sourcefps_metadata.csv",
  "tensor_file_mb": 34.3886775970459,
  "metadata_file_mb": 1.3333196640014648
}

Materializing finetune_val: 2839 rows
{
  "role": "finetune_val",
  "rows": 2839,
  "x_shape": [
    2839,
    3,
    240
  ],
  "y_shape": [
    2839
  ],
  "x_dtype": "float32",
  "y_dtype": "float32",
  "target_hr_min": 40.29643630981445,
  "target_hr_max": 127.0,
  "elapsed_seconds": 70.48568129400007,
  "seconds_per_row": 0.02482764399225

,role,rows,x_shape,y_shape,x_dtype,y_dtype,target_hr_min,target_hr_max,elapsed_seconds,seconds_per_row,tensor_path,metadata_path,tensor_file_mb,metadata_file_mb
0,finetune_train,12503,"(12503, 3, 240)","(12503,)",float32,float32,40.056313,127.000000,336.574241,0.026919,/kaggle/working/crvse_live_sourcefps_materiali...,/kaggle/working/crvse_live_sourcefps_materiali...,34.388678,1.333320
1,finetune_val,2839,"(2839, 3, 240)","(2839,)",float32,float32,40.296436,127.000000,70.485681,0.024828,/kaggle/working/crvse_live_sourcefps_materiali...,/kaggle/working/crvse_live_sourcefps_materiali...,7.808844,0.297821
2,finetune_test,2828,"(2828, 3, 240)","(2828,)",float32,float32,47.631721,129.071884,86.489532,0.030583,/kaggle/working/crvse_live_sourcefps_materiali...,/kaggle/working/crvse_live_sourcefps_materiali...,7.778589,0.293997
3,ood_eval,882,"(882, 3, 240)","(882,)",float32,float32,53.662189,167.968079,22.411902,0.025410,/kaggle/working/crvse_live_sourcefps_materiali...,/kaggle/working/crvse_live_sourcefps_materiali...,2.426317,0.075594



PASS: Full live-compatible source-FPS tensors were materialized successfully.


## Reload Materialized Tensors And Validate Cache

This section reloads the saved materialized tensors from `/kaggle/working`.

The goal is to verify that the training cache is usable before fine-tuning begins.

The notebook checks:

- tensor files can be loaded,
- metadata rows match tensor rows,
- PyTorch batches have shape `(batch, 3, 240)`,
- labels are finite,
- and frozen model predictions from cached tensors match the baseline CSV.

This protects the experiment from training on a corrupted or misaligned cache.

In [11]:
class MaterializedTensorDataset(Dataset):
    """PyTorch Dataset backed by precomputed source-FPS tensors and metadata."""

    def __init__(self, tensor_path: Path, metadata_path: Path) -> None:
        loaded = np.load(tensor_path)

        self.x = loaded["x"].astype(np.float32)
        self.y = loaded["y"].astype(np.float32)
        self.metadata = pd.read_csv(metadata_path)

        if len(self.x) != len(self.y):
            raise ValueError(f"x/y row mismatch: {len(self.x)} vs {len(self.y)}")

        if len(self.x) != len(self.metadata):
            raise ValueError(
                f"tensor/metadata row mismatch: {len(self.x)} vs {len(self.metadata)}"
            )

        if self.x.ndim != 3 or self.x.shape[1:] != (3, 240):
            raise ValueError(f"Expected x shape (n, 3, 240), found {self.x.shape}")

        if not np.isfinite(self.x).all():
            raise ValueError("x contains NaN or infinite values.")

        if not np.isfinite(self.y).all():
            raise ValueError("y contains NaN or infinite values.")

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, dict]:
        x = torch.tensor(self.x[index], dtype=torch.float32)
        y = torch.tensor(self.y[index], dtype=torch.float32)
        metadata = self.metadata.iloc[index].to_dict()
        return x, y, metadata


def load_materialized_dataset(role: str) -> MaterializedTensorDataset:
    """Load one materialized role dataset from the working cache."""
    tensor_path = MATERIALIZED_DIR / f"{role}_sourcefps_tensors.npz"
    metadata_path = MATERIALIZED_DIR / f"{role}_sourcefps_metadata.csv"

    if not tensor_path.exists():
        raise FileNotFoundError(tensor_path)

    if not metadata_path.exists():
        raise FileNotFoundError(metadata_path)

    return MaterializedTensorDataset(
        tensor_path=tensor_path,
        metadata_path=metadata_path,
    )


train_cached_ds = load_materialized_dataset("finetune_train")
val_cached_ds = load_materialized_dataset("finetune_val")
test_cached_ds = load_materialized_dataset("finetune_test")
ood_cached_ds = load_materialized_dataset("ood_eval")

train_cached_loader = DataLoader(
    train_cached_ds,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_with_metadata,
)

val_cached_loader = DataLoader(
    val_cached_ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_with_metadata,
)

cached_train_x, cached_train_y, cached_train_meta = next(iter(train_cached_loader))
cached_val_x, cached_val_y, cached_val_meta = next(iter(val_cached_loader))

cached_dataset_report = {
    "train_rows": len(train_cached_ds),
    "val_rows": len(val_cached_ds),
    "test_rows": len(test_cached_ds),
    "ood_rows": len(ood_cached_ds),
    "train_batch_x_shape": tuple(cached_train_x.shape),
    "train_batch_y_shape": tuple(cached_train_y.shape),
    "val_batch_x_shape": tuple(cached_val_x.shape),
    "val_batch_y_shape": tuple(cached_val_y.shape),
    "train_batch_finite": bool(torch.isfinite(cached_train_x).all().item()),
    "val_batch_finite": bool(torch.isfinite(cached_val_x).all().item()),
}

print("Cached dataset report:")
print(json.dumps(cached_dataset_report, indent=2))

print("\nCached validation metadata preview:")
display(pd.DataFrame(cached_val_meta).head())


def baseline_rows_for_metadata(metadata: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Find baseline prediction rows matching cached metadata."""
    selected_parts = []

    source = baseline_predictions.loc[
        baseline_predictions["preprocessing_mode"] == mode
    ].copy()

    source["subject_id"] = source["subject_id"].astype(str)
    source["recording_id"] = source["recording_id"].astype(str)
    source["dataset"] = source["dataset"].astype(str)
    source["start_frame"] = source["start_frame"].astype(int)
    source["end_frame"] = source["end_frame"].astype(int)

    for _, row in metadata.iterrows():
        mask = (
            (source["dataset"] == str(row["dataset"]))
            & (source["subject_id"] == str(row["subject_id"]))
            & (source["recording_id"] == str(row["recording_id"]))
            & (source["start_frame"] == int(row["start_frame"]))
            & (source["end_frame"] == int(row["end_frame"]))
        )

        matched = source.loc[mask]

        if len(matched) != 1:
            raise ValueError(
                "Expected exactly one baseline row for "
                f"{row['dataset']} {row['subject_id']} {row['recording_id']} "
                f"{row['start_frame']}:{row['end_frame']}, found {len(matched)}"
            )

        selected_parts.append(matched)

    return pd.concat(selected_parts, axis=0).reset_index(drop=True)


# Validate frozen predictions on the first cached validation batch.
model.eval()

cached_val_x_device = cached_val_x.to(DEVICE)

with torch.inference_mode():
    cached_val_predictions = model(cached_val_x_device).detach().cpu().numpy()

cached_val_metadata = pd.DataFrame(cached_val_meta)
cached_val_baseline = baseline_rows_for_metadata(
    metadata=cached_val_metadata,
    mode="training_buffer_source_fps",
)

cached_prediction_check = cached_val_metadata[
    [
        "dataset",
        "subject_id",
        "recording_id",
        "start_frame",
        "end_frame",
        "target_hr_mean",
    ]
].copy()

cached_prediction_check["notebook_model_hr"] = cached_val_predictions
cached_prediction_check["baseline_model_hr"] = pd.to_numeric(
    cached_val_baseline["model_hr"],
    errors="coerce",
)
cached_prediction_check["abs_diff_vs_baseline"] = (
    cached_prediction_check["notebook_model_hr"]
    - cached_prediction_check["baseline_model_hr"]
).abs()

print("\nCached validation batch prediction check:")
display(cached_prediction_check.head(20).round(4))

cache_prediction_report = {
    "checked_rows": int(len(cached_prediction_check)),
    "max_abs_diff_vs_baseline": float(cached_prediction_check["abs_diff_vs_baseline"].max()),
    "mean_abs_diff_vs_baseline": float(cached_prediction_check["abs_diff_vs_baseline"].mean()),
}

print("\nCache prediction report:")
print(json.dumps(cache_prediction_report, indent=2))

if cached_train_x.shape[1:] != (3, 240):
    raise AssertionError(f"Expected cached train batch tail shape (3, 240), found {cached_train_x.shape[1:]}")

if cached_val_x.shape[1:] != (3, 240):
    raise AssertionError(f"Expected cached val batch tail shape (3, 240), found {cached_val_x.shape[1:]}")

if cached_prediction_check["abs_diff_vs_baseline"].max() > 0.50:
    raise AssertionError("Cached tensor predictions do not match the frozen baseline closely enough.")

print("\nPASS: Materialized cache loads correctly and reproduces frozen source-FPS baseline predictions.")

Cached dataset report:
{
  "train_rows": 12503,
  "val_rows": 2839,
  "test_rows": 2828,
  "ood_rows": 882,
  "train_batch_x_shape": [
    64,
    3,
    240
  ],
  "train_batch_y_shape": [
    64
  ],
  "val_batch_x_shape": [
    64,
    3,
    240
  ],
  "val_batch_y_shape": [
    64
  ],
  "train_batch_finite": true,
  "val_batch_finite": true
}

Cached validation metadata preview:


,row_index,dataset,subject_id,recording_id,window_role,start_frame,end_frame,target_hr_mean,label_stability_bucket,source_fps,actual_buffer_seconds,buffer_sample_count
0,0,mcd_rppg,1115,after,finetune_val,0,239,93.837090,moderate_10_25_bpm,29.9,7.993311,239
1,1,mcd_rppg,1115,after,finetune_val,119,358,94.842476,moderate_10_25_bpm,29.9,11.973244,358
2,2,mcd_rppg,1115,after,finetune_val,238,477,93.438217,moderate_10_25_bpm,29.9,11.973244,358
3,3,mcd_rppg,1115,after,finetune_val,357,596,93.003090,moderate_10_25_bpm,29.9,11.973244,358
4,4,mcd_rppg,1115,after,finetune_val,476,715,92.221169,moderate_10_25_bpm,29.9,11.973244,358



Cached validation batch prediction check:


,dataset,subject_id,recording_id,start_frame,end_frame,target_hr_mean,notebook_model_hr,baseline_model_hr,abs_diff_vs_baseline
0,mcd_rppg,1115,after,0,239,93.8371,103.092697,103.0927,0.0
1,mcd_rppg,1115,after,119,358,94.8425,87.599602,87.5996,0.0
2,mcd_rppg,1115,after,238,477,93.4382,86.780403,86.7804,0.0
3,mcd_rppg,1115,after,357,596,93.0031,96.860802,96.8607,0.0
4,mcd_rppg,1115,after,476,715,92.2212,92.439301,92.4393,0.0
5,mcd_rppg,1115,after,595,834,91.6968,92.331398,92.3314,0.0
6,mcd_rppg,1115,after,714,953,95.1685,98.005898,98.0059,0.0
7,mcd_rppg,1115,after,833,1072,97.5786,93.074402,93.0744,0.0
8,mcd_rppg,1115,after,952,1191,94.5506,92.879303,92.8793,0.0
9,mcd_rppg,1115,after,1071,1310,91.1603,101.069397,101.0694,0.0



Cache prediction report:
{
  "checked_rows": 64,
  "max_abs_diff_vs_baseline": 3.814697265625e-05,
  "mean_abs_diff_vs_baseline": 1.0251998902033321e-05
}

PASS: Materialized cache loads correctly and reproduces frozen source-FPS baseline predictions.


## Full Frozen Evaluation From Materialized Cache

This section evaluates the frozen checkpoint on the full materialized cache.

The goal is to verify the evaluation function before fine-tuning begins.

The expected source-FPS frozen baselines are approximately:

```text
finetune_val  mean MAE ~6.03 BPM
finetune_test mean MAE ~5.77 BPM
ood_eval      mean MAE ~14.04 BPM

In [12]:
def make_cached_loader(
    dataset: Dataset,
    batch_size: int = 256,
    shuffle: bool = False,
) -> DataLoader:
    """Create a DataLoader for a materialized tensor dataset."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=collate_with_metadata,
        pin_memory=(DEVICE.type == "cuda"),
    )


def evaluate_hr_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> tuple[dict, pd.DataFrame]:
    """Evaluate HR model and return aggregate metrics plus per-window predictions."""
    model.eval()

    rows = []
    abs_errors = []

    with torch.inference_mode():
        for x, y, metadata in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            predictions = model(x)
            errors = torch.abs(predictions - y)

            predictions_np = predictions.detach().cpu().numpy()
            targets_np = y.detach().cpu().numpy()
            errors_np = errors.detach().cpu().numpy()

            for idx, meta in enumerate(metadata):
                row = dict(meta)
                row["target_hr"] = float(targets_np[idx])
                row["model_hr"] = float(predictions_np[idx])
                row["abs_error"] = float(errors_np[idx])
                rows.append(row)

            abs_errors.extend(errors_np.tolist())

    abs_errors = np.asarray(abs_errors, dtype=np.float32)

    metrics = {
        "rows": int(len(abs_errors)),
        "mae_mean": float(np.mean(abs_errors)),
        "mae_median": float(np.median(abs_errors)),
        "mae_p90": float(np.quantile(abs_errors, 0.90)),
        "mae_p95": float(np.quantile(abs_errors, 0.95)),
        "mae_max": float(np.max(abs_errors)),
    }

    return metrics, pd.DataFrame(rows)


cached_loaders = {
    "finetune_train": make_cached_loader(train_cached_ds, batch_size=256, shuffle=False),
    "finetune_val": make_cached_loader(val_cached_ds, batch_size=256, shuffle=False),
    "finetune_test": make_cached_loader(test_cached_ds, batch_size=256, shuffle=False),
    "ood_eval": make_cached_loader(ood_cached_ds, batch_size=256, shuffle=False),
}

frozen_eval_metrics = {}
frozen_eval_predictions = {}

for role, loader in cached_loaders.items():
    metrics, predictions_df = evaluate_hr_model(
        model=model,
        loader=loader,
        device=DEVICE,
    )
    frozen_eval_metrics[role] = metrics
    frozen_eval_predictions[role] = predictions_df

frozen_eval_table = (
    pd.DataFrame.from_dict(frozen_eval_metrics, orient="index")
    .reset_index()
    .rename(columns={"index": "role"})
)

for col in ["mae_mean", "mae_median", "mae_p90", "mae_p95", "mae_max"]:
    frozen_eval_table[col] = frozen_eval_table[col].round(4)

print("Frozen full-cache evaluation:")
display(frozen_eval_table)

frozen_by_dataset_role = []

for role, predictions_df in frozen_eval_predictions.items():
    grouped = (
        predictions_df
        .groupby(["dataset", "window_role"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            target_hr_mean=("target_hr", "mean"),
        )
        .reset_index()
    )
    frozen_by_dataset_role.append(grouped)

frozen_by_dataset_role = pd.concat(frozen_by_dataset_role, axis=0).reset_index(drop=True)

for col in ["mae_mean", "mae_median", "mae_p90", "target_hr_mean"]:
    frozen_by_dataset_role[col] = frozen_by_dataset_role[col].round(4)

print("\nFrozen full-cache evaluation by dataset and role:")
display(frozen_by_dataset_role.sort_values(["window_role", "dataset"]))

expected_frozen_mae = {
    "finetune_val": 6.03,
    "finetune_test": 5.77,
    "ood_eval": 14.04,
}

tolerance_bpm = 0.10

for role, expected_mae in expected_frozen_mae.items():
    actual_mae = float(frozen_eval_metrics[role]["mae_mean"])
    if abs(actual_mae - expected_mae) > tolerance_bpm:
        raise AssertionError(
            f"{role} frozen MAE differs from expected baseline: "
            f"actual={actual_mae:.4f}, expected≈{expected_mae:.4f}"
        )

print("\nPASS: Full cached frozen evaluation reproduces the expected source-FPS baseline.")

Frozen full-cache evaluation:


,role,rows,mae_mean,mae_median,mae_p90,mae_p95,mae_max
0,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838
1,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965
2,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401
3,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549



Frozen full-cache evaluation by dataset and role:


,dataset,window_role,rows,mae_mean,mae_median,mae_p90,target_hr_mean
6,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.8533
7,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.4205
8,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.2693
0,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.0648
1,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.3055
2,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.4492
3,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.1086
4,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.0024
5,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.2772
9,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.2048



PASS: Full cached frozen evaluation reproduces the expected source-FPS baseline.


## Head-Only Fine-Tuning

This section runs the first real fine-tuning experiment.

Only the final regression head is trainable. The feature extractor, FFT projection, positional encoding, and transformer layers remain frozen.

This is the smallest safe weight update because it tests whether the current representation already contains useful information for the live-compatible source-FPS input distribution.

Model selection uses validation MAE, not training loss.

A useful result would be:

```text
finetune_val MAE improves below the frozen baseline
finetune_test MAE does not degrade
OOD ECG-Fitness is reported separately

In [13]:
HEAD_ONLY_EPOCHS = 30
HEAD_ONLY_BATCH_SIZE = 256
HEAD_ONLY_LR = 1e-4
HEAD_ONLY_WEIGHT_DECAY = 1e-4
HEAD_ONLY_PATIENCE = 6
HEAD_ONLY_MIN_DELTA = 0.01
GRAD_CLIP_NORM = 1.0

HEAD_ONLY_DIR = KAGGLE_WORKING / "crvse_head_only_sourcefps"
HEAD_ONLY_DIR.mkdir(parents=True, exist_ok=True)


def freeze_for_head_only(model_to_train: nn.Module) -> list[nn.Parameter]:
    """Freeze all parameters except the final regression head."""
    for parameter in model_to_train.parameters():
        parameter.requires_grad = False

    for parameter in model_to_train.head.parameters():
        parameter.requires_grad = True

    return [parameter for parameter in model_to_train.parameters() if parameter.requires_grad]


def set_head_only_train_mode(model_to_train: nn.Module) -> None:
    """Keep frozen feature modules deterministic while training the head."""
    model_to_train.train()

    model_to_train.encoder.eval()
    model_to_train.freq_proj.eval()
    model_to_train.pos_enc.eval()

    for layer in model_to_train.transformer_layers:
        layer.eval()

    model_to_train.head.train()


def parameter_summary(model_to_train: nn.Module) -> pd.DataFrame:
    """Summarize trainable and frozen parameters by top-level module."""
    rows = []

    for name, parameter in model_to_train.named_parameters():
        top_module = name.split(".")[0]
        rows.append(
            {
                "module": top_module,
                "parameters": int(parameter.numel()),
                "trainable": bool(parameter.requires_grad),
            }
        )

    summary = (
        pd.DataFrame(rows)
        .groupby(["module", "trainable"], dropna=False)
        .agg(parameters=("parameters", "sum"))
        .reset_index()
        .sort_values(["module", "trainable"])
    )

    return summary


def train_head_only_one_epoch(
    model_to_train: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
    trainable_parameters: list[nn.Parameter],
) -> dict:
    """Train the regression head for one epoch and return training metrics."""
    set_head_only_train_mode(model_to_train)

    total_loss = 0.0
    total_abs_error = 0.0
    total_rows = 0
    grad_norms = []

    for x, y, _metadata in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        predictions = model_to_train(x)
        loss = loss_fn(predictions, y)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=GRAD_CLIP_NORM,
        )
        optimizer.step()

        batch_size = int(y.shape[0])
        total_rows += batch_size
        total_loss += float(loss.detach().cpu().item()) * batch_size
        total_abs_error += float(torch.sum(torch.abs(predictions.detach() - y)).cpu().item())
        grad_norms.append(float(grad_norm.detach().cpu().item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_abs_error / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


head_train_loader = make_cached_loader(
    train_cached_ds,
    batch_size=HEAD_ONLY_BATCH_SIZE,
    shuffle=True,
)

head_val_loader = make_cached_loader(
    val_cached_ds,
    batch_size=HEAD_ONLY_BATCH_SIZE,
    shuffle=False,
)

head_only_model = copy.deepcopy(model).to(DEVICE)
trainable_parameters = freeze_for_head_only(head_only_model)

loss_fn = nn.HuberLoss(delta=float(best_params["huber_delta"]))
optimizer = AdamW(
    trainable_parameters,
    lr=HEAD_ONLY_LR,
    weight_decay=HEAD_ONLY_WEIGHT_DECAY,
)

frozen_val_mae = float(frozen_eval_metrics["finetune_val"]["mae_mean"])
best_val_mae = frozen_val_mae
best_epoch = 0
best_state = copy.deepcopy(head_only_model.state_dict())
epochs_without_improvement = 0

history_rows = []

print("Head-only parameter summary:")
display(parameter_summary(head_only_model))

print(f"Frozen validation MAE baseline: {frozen_val_mae:.4f} BPM")

for epoch in range(1, HEAD_ONLY_EPOCHS + 1):
    train_metrics = train_head_only_one_epoch(
        model_to_train=head_only_model,
        loader=head_train_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        trainable_parameters=trainable_parameters,
    )

    val_metrics, _val_predictions = evaluate_hr_model(
        model=head_only_model,
        loader=head_val_loader,
        device=DEVICE,
    )

    current_val_mae = float(val_metrics["mae_mean"])

    improved = current_val_mae < (best_val_mae - HEAD_ONLY_MIN_DELTA)

    if improved:
        best_val_mae = current_val_mae
        best_epoch = epoch
        best_state = copy.deepcopy(head_only_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["train_loss"],
            "train_mae": train_metrics["train_mae"],
            "val_mae": current_val_mae,
            "val_median": float(val_metrics["mae_median"]),
            "val_p90": float(val_metrics["mae_p90"]),
            "grad_norm_mean": train_metrics["grad_norm_mean"],
            "grad_norm_max": train_metrics["grad_norm_max"],
            "improved": improved,
            "best_epoch_so_far": best_epoch,
            "best_val_mae_so_far": best_val_mae,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train MAE {train_metrics['train_mae']:.4f} | "
        f"val MAE {current_val_mae:.4f} | "
        f"best {best_val_mae:.4f} @ epoch {best_epoch}"
    )

    if epochs_without_improvement >= HEAD_ONLY_PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

head_only_history = pd.DataFrame(history_rows)

head_only_model.load_state_dict(best_state)
head_only_model.eval()

head_only_eval_metrics = {}
head_only_eval_predictions = {}

for role, loader in cached_loaders.items():
    metrics, predictions_df = evaluate_hr_model(
        model=head_only_model,
        loader=loader,
        device=DEVICE,
    )
    head_only_eval_metrics[role] = metrics
    head_only_eval_predictions[role] = predictions_df

comparison_rows = []

for role, metrics in head_only_eval_metrics.items():
    frozen_metrics = frozen_eval_metrics[role]

    comparison_rows.append(
        {
            "role": role,
            "rows": int(metrics["rows"]),
            "frozen_mae_mean": float(frozen_metrics["mae_mean"]),
            "head_only_mae_mean": float(metrics["mae_mean"]),
            "delta_mae_mean": float(metrics["mae_mean"] - frozen_metrics["mae_mean"]),
            "frozen_mae_median": float(frozen_metrics["mae_median"]),
            "head_only_mae_median": float(metrics["mae_median"]),
            "delta_mae_median": float(metrics["mae_median"] - frozen_metrics["mae_median"]),
            "frozen_mae_p90": float(frozen_metrics["mae_p90"]),
            "head_only_mae_p90": float(metrics["mae_p90"]),
            "delta_mae_p90": float(metrics["mae_p90"] - frozen_metrics["mae_p90"]),
        }
    )

head_only_comparison = pd.DataFrame(comparison_rows)

metric_cols = [
    "frozen_mae_mean",
    "head_only_mae_mean",
    "delta_mae_mean",
    "frozen_mae_median",
    "head_only_mae_median",
    "delta_mae_median",
    "frozen_mae_p90",
    "head_only_mae_p90",
    "delta_mae_p90",
]

head_only_comparison[metric_cols] = head_only_comparison[metric_cols].round(4)

history_path = HEAD_ONLY_DIR / "head_only_sourcefps_history.csv"
comparison_path = HEAD_ONLY_DIR / "head_only_sourcefps_comparison.csv"
checkpoint_path = HEAD_ONLY_DIR / "head_only_sourcefps_best.pt"

head_only_history.to_csv(history_path, index=False)
head_only_comparison.to_csv(comparison_path, index=False)

torch.save(
    {
        "experiment": "head_only_sourcefps_live_compatible",
        "base_checkpoint_path": str(CHECKPOINT_PATH),
        "trainable_mode": "head_only",
        "model_state": head_only_model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_mae": best_val_mae,
        "frozen_val_mae": frozen_val_mae,
        "history": head_only_history.to_dict("records"),
        "comparison": head_only_comparison.to_dict("records"),
        "config": {
            "epochs": HEAD_ONLY_EPOCHS,
            "batch_size": HEAD_ONLY_BATCH_SIZE,
            "lr": HEAD_ONLY_LR,
            "weight_decay": HEAD_ONLY_WEIGHT_DECAY,
            "patience": HEAD_ONLY_PATIENCE,
            "min_delta": HEAD_ONLY_MIN_DELTA,
            "huber_delta": float(best_params["huber_delta"]),
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "checkpoint_metadata": {
            "input_mode": checkpoint.get("input_mode"),
            "in_channels": checkpoint.get("in_channels"),
            "best_params": checkpoint.get("best_params"),
        },
    },
    checkpoint_path,
)

print("\nHead-only training history:")
display(head_only_history.round(4))

print("\nHead-only vs frozen comparison:")
display(head_only_comparison)

result_summary = {
    "best_epoch": int(best_epoch),
    "frozen_val_mae": float(frozen_val_mae),
    "best_val_mae": float(best_val_mae),
    "val_delta_vs_frozen": float(best_val_mae - frozen_val_mae),
    "history_path": str(history_path),
    "comparison_path": str(comparison_path),
    "checkpoint_path": str(checkpoint_path),
}

print("\nHead-only result summary:")
print(json.dumps(result_summary, indent=2))

if best_epoch == 0:
    print("\nRESULT: Head-only fine-tuning did not beat the frozen validation baseline.")
else:
    print("\nRESULT: Head-only fine-tuning improved validation MAE versus the frozen baseline.")

print("\nDONE: Head-only fine-tuning experiment completed.")

Head-only parameter summary:


,module,trainable,parameters
0,encoder,False,1696
1,freq_proj,False,47680
2,head,True,2625
3,transformer_layers,False,270144


Frozen validation MAE baseline: 6.0275 BPM
Epoch 01 | train MAE 8.6340 | val MAE 6.1032 | best 6.0275 @ epoch 0
Epoch 02 | train MAE 8.6462 | val MAE 6.1150 | best 6.0275 @ epoch 0
Epoch 03 | train MAE 8.6429 | val MAE 6.0975 | best 6.0275 @ epoch 0
Epoch 04 | train MAE 8.6517 | val MAE 6.0047 | best 6.0047 @ epoch 4
Epoch 05 | train MAE 8.6080 | val MAE 6.0848 | best 6.0047 @ epoch 4
Epoch 06 | train MAE 8.6409 | val MAE 6.0551 | best 6.0047 @ epoch 4
Epoch 07 | train MAE 8.6096 | val MAE 6.1057 | best 6.0047 @ epoch 4
Epoch 08 | train MAE 8.6309 | val MAE 6.0974 | best 6.0047 @ epoch 4
Epoch 09 | train MAE 8.6254 | val MAE 6.0986 | best 6.0047 @ epoch 4
Epoch 10 | train MAE 8.6458 | val MAE 6.1091 | best 6.0047 @ epoch 4
Early stopping after 10 epochs.

Head-only training history:


,epoch,train_loss,train_mae,val_mae,val_median,val_p90,grad_norm_mean,grad_norm_max,improved,best_epoch_so_far,best_val_mae_so_far
0,1,38.8601,8.6340,6.1032,4.0939,14.4077,103.2321,306.0842,False,0,6.0275
1,2,38.8619,8.6462,6.1150,4.1046,14.3665,94.2131,262.6241,False,0,6.0275
2,3,38.7794,8.6429,6.0975,4.1008,14.3517,80.1908,309.5166,False,0,6.0275
3,4,38.8570,8.6517,6.0047,4.0423,14.3648,89.3274,245.8683,True,4,6.0047
4,5,38.6884,8.6080,6.0848,4.0968,14.3125,79.0748,195.3176,False,4,6.0047
5,6,38.8068,8.6409,6.0551,4.0677,14.3230,93.1061,290.5962,False,4,6.0047
6,7,38.6694,8.6096,6.1057,4.1340,14.3489,74.9806,244.8563,False,4,6.0047
7,8,38.7819,8.6309,6.0974,4.1347,14.3339,78.0164,240.4739,False,4,6.0047
8,9,38.7623,8.6254,6.0986,4.1519,14.3361,73.0776,243.7330,False,4,6.0047
9,10,38.8768,8.6458,6.1091,4.1586,14.3560,76.4109,210.2345,False,4,6.0047



Head-only vs frozen comparison:


,role,rows,frozen_mae_mean,head_only_mae_mean,delta_mae_mean,frozen_mae_median,head_only_mae_median,delta_mae_median,frozen_mae_p90,head_only_mae_p90,delta_mae_p90
0,finetune_train,12503,5.8236,5.8359,0.0123,3.7437,3.9703,0.2266,13.7192,13.5125,-0.2067
1,finetune_val,2839,6.0275,6.0047,-0.0228,3.8533,4.0423,0.1890,14.0701,14.3648,0.2947
2,finetune_test,2828,5.7696,5.7841,0.0144,3.6274,3.6789,0.0516,13.6984,13.3458,-0.3526
3,ood_eval,882,14.0355,14.7259,0.6904,7.6838,7.9928,0.3089,38.1498,40.6134,2.4636



Head-only result summary:
{
  "best_epoch": 4,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 6.004687786102295,
  "val_delta_vs_frozen": -0.022814273834228516,
  "history_path": "/kaggle/working/crvse_head_only_sourcefps/head_only_sourcefps_history.csv",
  "comparison_path": "/kaggle/working/crvse_head_only_sourcefps/head_only_sourcefps_comparison.csv",
  "checkpoint_path": "/kaggle/working/crvse_head_only_sourcefps/head_only_sourcefps_best.pt"
}

RESULT: Head-only fine-tuning improved validation MAE versus the frozen baseline.

DONE: Head-only fine-tuning experiment completed.


## Head-Only Error Analysis By Dataset And Stability

This section checks where head-only fine-tuning helped or hurt.

The validation mean MAE improved only slightly, while test and out-of-domain metrics worsened. Before unfreezing more of the model, the notebook analyzes error changes by:

- role,
- dataset,
- and label-stability bucket.

This helps decide whether the next experiment should unfreeze the last transformer block plus the head, or whether the current fine-tuning objective needs adjustment.

In [14]:
def summarize_prediction_errors(predictions_by_role: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Summarize prediction errors by role, dataset, and label-stability bucket."""
    parts = []

    for role, predictions_df in predictions_by_role.items():
        grouped = (
            predictions_df
            .groupby(["window_role", "dataset", "label_stability_bucket"], dropna=False)
            .agg(
                rows=("abs_error", "size"),
                mae_mean=("abs_error", "mean"),
                mae_median=("abs_error", "median"),
                mae_p90=("abs_error", lambda s: float(s.quantile(0.90))),
                target_hr_mean=("target_hr", "mean"),
            )
            .reset_index()
        )
        parts.append(grouped)

    summary = pd.concat(parts, axis=0).reset_index(drop=True)

    for col in ["mae_mean", "mae_median", "mae_p90", "target_hr_mean"]:
        summary[col] = summary[col].round(4)

    return summary


frozen_error_summary = summarize_prediction_errors(frozen_eval_predictions)
head_only_error_summary = summarize_prediction_errors(head_only_eval_predictions)

error_change = frozen_error_summary.merge(
    head_only_error_summary,
    on=["window_role", "dataset", "label_stability_bucket"],
    suffixes=("_frozen", "_head_only"),
)

error_change["delta_mae_mean"] = (
    error_change["mae_mean_head_only"] - error_change["mae_mean_frozen"]
).round(4)

error_change["delta_mae_median"] = (
    error_change["mae_median_head_only"] - error_change["mae_median_frozen"]
).round(4)

error_change["delta_mae_p90"] = (
    error_change["mae_p90_head_only"] - error_change["mae_p90_frozen"]
).round(4)

display_cols = [
    "window_role",
    "dataset",
    "label_stability_bucket",
    "rows_frozen",
    "mae_mean_frozen",
    "mae_mean_head_only",
    "delta_mae_mean",
    "mae_median_frozen",
    "mae_median_head_only",
    "delta_mae_median",
    "mae_p90_frozen",
    "mae_p90_head_only",
    "delta_mae_p90",
]

print("Head-only error change by role, dataset, and stability:")
display(
    error_change[display_cols]
    .sort_values(["window_role", "dataset", "label_stability_bucket"])
)

print("\nGroups most improved by head-only fine-tuning:")
display(
    error_change[display_cols]
    .sort_values("delta_mae_mean")
    .head(10)
)

print("\nGroups most worsened by head-only fine-tuning:")
display(
    error_change[display_cols]
    .sort_values("delta_mae_mean", ascending=False)
    .head(10)
)

head_only_decision = {
    "val_mean_delta_bpm": float(
        head_only_comparison.loc[
            head_only_comparison["role"] == "finetune_val",
            "delta_mae_mean",
        ].iloc[0]
    ),
    "test_mean_delta_bpm": float(
        head_only_comparison.loc[
            head_only_comparison["role"] == "finetune_test",
            "delta_mae_mean",
        ].iloc[0]
    ),
    "ood_mean_delta_bpm": float(
        head_only_comparison.loc[
            head_only_comparison["role"] == "ood_eval",
            "delta_mae_mean",
        ].iloc[0]
    ),
    "decision": (
        "Do not adopt head-only checkpoint as final. "
        "Use it as evidence that the head alone is too weak; "
        "next experiment should test last-block-plus-head fine-tuning."
    ),
}

print("\nHead-only decision:")
print(json.dumps(head_only_decision, indent=2))

Head-only error change by role, dataset, and stability:


,window_role,dataset,label_stability_bucket,rows_frozen,mae_mean_frozen,mae_mean_head_only,delta_mae_mean,mae_median_frozen,mae_median_head_only,delta_mae_median,mae_p90_frozen,mae_p90_head_only,delta_mae_p90
22,finetune_test,mcd_rppg,highly_unstable_ge_50_bpm,52,10.0104,9.6423,-0.3681,7.3807,6.5441,-0.8366,24.7500,21.8144,-2.9356
23,finetune_test,mcd_rppg,moderate_10_25_bpm,825,5.2657,5.2947,0.0290,3.5729,3.5753,0.0024,12.0972,12.5374,0.4402
24,finetune_test,mcd_rppg,stable_lt_10_bpm,1109,4.1016,4.1426,0.0410,2.6794,2.8295,0.1501,9.1908,8.6178,-0.5730
25,finetune_test,mcd_rppg,unstable_25_50_bpm,108,7.5483,7.4176,-0.1307,6.0920,4.8698,-1.2222,16.4021,16.1690,-0.2331
26,finetune_test,ubfc_phys,highly_unstable_ge_50_bpm,494,10.3987,10.0411,-0.3576,7.7465,7.4958,-0.2507,22.1340,20.5668,-1.5672
27,finetune_test,ubfc_phys,moderate_10_25_bpm,42,3.1211,4.4027,1.2816,2.2275,4.1387,1.9112,6.4105,8.6155,2.2050
28,finetune_test,ubfc_phys,stable_lt_10_bpm,4,1.8013,1.8975,0.0962,1.6917,1.0083,-0.6834,2.8420,3.7191,0.8771
29,finetune_test,ubfc_phys,unstable_25_50_bpm,118,4.6916,5.2120,0.5204,3.3532,3.7083,0.3551,11.8280,11.9557,0.1277
30,finetune_test,ubfc_rppg,moderate_10_25_bpm,7,4.1079,2.7168,-1.3911,3.6695,2.7348,-0.9347,5.9837,4.7848,-1.1989
31,finetune_test,ubfc_rppg,stable_lt_10_bpm,69,3.3375,4.4316,1.0941,3.0262,3.7688,0.7426,5.9522,8.5130,2.5608



Groups most improved by head-only fine-tuning:


,window_role,dataset,label_stability_bucket,rows_frozen,mae_mean_frozen,mae_mean_head_only,delta_mae_mean,mae_median_frozen,mae_median_head_only,delta_mae_median,mae_p90_frozen,mae_p90_head_only,delta_mae_p90
10,finetune_train,ubfc_rppg,unstable_25_50_bpm,3,12.8766,9.4836,-3.3930,16.5014,12.9236,-3.5778,16.7005,13.7367,-2.9638
30,finetune_test,ubfc_rppg,moderate_10_25_bpm,7,4.1079,2.7168,-1.3911,3.6695,2.7348,-0.9347,5.9837,4.7848,-1.1989
0,finetune_train,mcd_rppg,highly_unstable_ge_50_bpm,199,7.8946,7.4738,-0.4208,6.0866,5.1390,-0.9476,17.1443,15.3889,-1.7554
13,finetune_val,mcd_rppg,stable_lt_10_bpm,771,4.5882,4.1982,-0.3900,2.9414,2.5368,-0.4046,10.2410,9.8470,-0.3940
22,finetune_test,mcd_rppg,highly_unstable_ge_50_bpm,52,10.0104,9.6423,-0.3681,7.3807,6.5441,-0.8366,24.7500,21.8144,-2.9356
26,finetune_test,ubfc_phys,highly_unstable_ge_50_bpm,494,10.3987,10.0411,-0.3576,7.7465,7.4958,-0.2507,22.1340,20.5668,-1.5672
4,finetune_train,ubfc_phys,highly_unstable_ge_50_bpm,2831,8.7302,8.4520,-0.2782,6.6318,6.6018,-0.0300,19.5029,18.2702,-1.2327
15,finetune_val,ubfc_phys,highly_unstable_ge_50_bpm,614,9.5030,9.2452,-0.2578,6.7600,6.6502,-0.1098,21.3904,19.5483,-1.8421
2,finetune_train,mcd_rppg,stable_lt_10_bpm,3176,4.3666,4.2277,-0.1389,2.8408,2.8061,-0.0347,9.7049,9.2808,-0.4241
25,finetune_test,mcd_rppg,unstable_25_50_bpm,108,7.5483,7.4176,-0.1307,6.0920,4.8698,-1.2222,16.4021,16.1690,-0.2331



Groups most worsened by head-only fine-tuning:


,window_role,dataset,label_stability_bucket,rows_frozen,mae_mean_frozen,mae_mean_head_only,delta_mae_mean,mae_median_frozen,mae_median_head_only,delta_mae_median,mae_p90_frozen,mae_p90_head_only,delta_mae_p90
17,finetune_val,ubfc_phys,stable_lt_10_bpm,65,2.8850,4.9087,2.0237,2.3624,4.6691,2.3067,5.7209,8.1744,2.4535
20,finetune_val,ubfc_rppg,moderate_10_25_bpm,11,4.8904,6.4573,1.5669,1.7563,4.1382,2.3819,10.6193,13.3654,2.7461
21,finetune_val,ubfc_rppg,stable_lt_10_bpm,97,4.6932,6.1284,1.4352,2.4882,4.3383,1.8501,12.7234,15.5523,2.8289
27,finetune_test,ubfc_phys,moderate_10_25_bpm,42,3.1211,4.4027,1.2816,2.2275,4.1387,1.9112,6.4105,8.6155,2.2050
9,finetune_train,ubfc_rppg,stable_lt_10_bpm,425,4.3584,5.5117,1.1533,2.9342,4.4747,1.5405,9.5181,11.4674,1.9493
31,finetune_test,ubfc_rppg,stable_lt_10_bpm,69,3.3375,4.4316,1.0941,3.0262,3.7688,0.7426,5.9522,8.5130,2.5608
35,ood_eval,ecg_fitness,unstable_25_50_bpm,25,10.2919,11.3236,1.0317,3.7663,5.7050,1.9387,26.2144,28.7311,2.5167
6,finetune_train,ubfc_phys,stable_lt_10_bpm,168,5.6408,6.4749,0.8341,2.5886,3.9735,1.3849,13.7274,14.8411,1.1137
5,finetune_train,ubfc_phys,moderate_10_25_bpm,637,5.4588,6.2655,0.8067,3.5179,4.3861,0.8682,13.5940,14.4647,0.8707
33,ood_eval,ecg_fitness,moderate_10_25_bpm,200,12.6636,13.4597,0.7961,7.3831,8.0720,0.6889,31.5596,34.2310,2.6714



Head-only decision:
{
  "val_mean_delta_bpm": -0.0228,
  "test_mean_delta_bpm": 0.0144,
  "ood_mean_delta_bpm": 0.6904,
  "decision": "Do not adopt head-only checkpoint as final. Use it as evidence that the head alone is too weak; next experiment should test last-block-plus-head fine-tuning."
}


## Last-Block Plus Head Fine-Tuning

This section tests a slightly less restrictive fine-tuning strategy.

The model keeps most of the checkpoint frozen:

- CNN encoder remains frozen.
- FFT projection remains frozen.
- Positional encoding remains frozen.
- Earlier transformer layers remain frozen.

Only the final transformer block and regression head are trainable.

This tests whether the live-compatible source-FPS distribution requires a small adaptation of the learned temporal representation, rather than only a new output calibration.

The experiment starts from the original frozen checkpoint, not from the head-only checkpoint.

A useful result should improve validation MAE meaningfully and should not degrade held-out test performance. ECG-Fitness remains out-of-domain evaluation and is reported separately.

In [15]:
LAST_BLOCK_EPOCHS = 30
LAST_BLOCK_BATCH_SIZE = 256
LAST_BLOCK_LR = 1e-5
LAST_BLOCK_HEAD_LR = 5e-5
LAST_BLOCK_WEIGHT_DECAY = 1e-4
LAST_BLOCK_PATIENCE = 7
LAST_BLOCK_MIN_DELTA = 0.02

LAST_BLOCK_DIR = KAGGLE_WORKING / "crvse_last_block_head_sourcefps"
LAST_BLOCK_DIR.mkdir(parents=True, exist_ok=True)


def freeze_for_last_block_plus_head(model_to_train: nn.Module) -> list[nn.Parameter]:
    """Freeze all parameters except the final transformer block and regression head."""
    for parameter in model_to_train.parameters():
        parameter.requires_grad = False

    for parameter in model_to_train.transformer_layers[-1].parameters():
        parameter.requires_grad = True

    for parameter in model_to_train.head.parameters():
        parameter.requires_grad = True

    return [p for p in model_to_train.parameters() if p.requires_grad]


def set_last_block_plus_head_train_mode(model_to_train: nn.Module) -> None:
    """Train only the final transformer block and head while frozen modules stay deterministic."""
    model_to_train.train()

    model_to_train.encoder.eval()
    model_to_train.freq_proj.eval()
    model_to_train.pos_enc.eval()

    for layer in model_to_train.transformer_layers[:-1]:
        layer.eval()

    model_to_train.transformer_layers[-1].train()
    model_to_train.head.train()


def train_last_block_epoch(
    model_to_train: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
    trainable_parameters: list[nn.Parameter],
) -> dict:
    """Train last block plus head for one epoch."""
    set_last_block_plus_head_train_mode(model_to_train)

    total_loss = 0.0
    total_abs_error = 0.0
    total_rows = 0
    grad_norms = []

    for x, y, _metadata in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        predictions = model_to_train(x)
        loss = loss_fn(predictions, y)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=GRAD_CLIP_NORM,
        )
        optimizer.step()

        batch_size = int(y.shape[0])
        total_rows += batch_size
        total_loss += float(loss.detach().cpu().item()) * batch_size
        total_abs_error += float(torch.sum(torch.abs(predictions.detach() - y)).cpu().item())
        grad_norms.append(float(grad_norm.detach().cpu().item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_abs_error / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def compare_eval_to_frozen(candidate_metrics: dict, frozen_metrics: dict) -> pd.DataFrame:
    """Compare candidate evaluation metrics against frozen baseline metrics."""
    rows = []

    for role, metrics in candidate_metrics.items():
        frozen = frozen_metrics[role]
        rows.append(
            {
                "role": role,
                "rows": int(metrics["rows"]),
                "frozen_mae_mean": float(frozen["mae_mean"]),
                "candidate_mae_mean": float(metrics["mae_mean"]),
                "delta_mae_mean": float(metrics["mae_mean"] - frozen["mae_mean"]),
                "frozen_mae_median": float(frozen["mae_median"]),
                "candidate_mae_median": float(metrics["mae_median"]),
                "delta_mae_median": float(metrics["mae_median"] - frozen["mae_median"]),
                "frozen_mae_p90": float(frozen["mae_p90"]),
                "candidate_mae_p90": float(metrics["mae_p90"]),
                "delta_mae_p90": float(metrics["mae_p90"] - frozen["mae_p90"]),
            }
        )

    table = pd.DataFrame(rows)
    metric_cols = [col for col in table.columns if col not in ["role", "rows"]]
    table[metric_cols] = table[metric_cols].round(4)
    return table


last_block_train_loader = make_cached_loader(
    train_cached_ds,
    batch_size=LAST_BLOCK_BATCH_SIZE,
    shuffle=True,
)

last_block_val_loader = make_cached_loader(
    val_cached_ds,
    batch_size=LAST_BLOCK_BATCH_SIZE,
    shuffle=False,
)

last_block_model = copy.deepcopy(model).to(DEVICE)
trainable_parameters = freeze_for_last_block_plus_head(last_block_model)

last_block_params = list(last_block_model.transformer_layers[-1].parameters())
head_params = list(last_block_model.head.parameters())

optimizer = AdamW(
    [
        {"params": last_block_params, "lr": LAST_BLOCK_LR},
        {"params": head_params, "lr": LAST_BLOCK_HEAD_LR},
    ],
    weight_decay=LAST_BLOCK_WEIGHT_DECAY,
)

loss_fn = nn.HuberLoss(delta=float(best_params["huber_delta"]))

frozen_val_mae = float(frozen_eval_metrics["finetune_val"]["mae_mean"])
best_val_mae = frozen_val_mae
best_epoch = 0
best_state = copy.deepcopy(last_block_model.state_dict())
epochs_without_improvement = 0
history_rows = []

print("Last-block plus head parameter summary:")
display(parameter_summary(last_block_model))

print(f"Frozen validation MAE baseline: {frozen_val_mae:.4f} BPM")

for epoch in range(1, LAST_BLOCK_EPOCHS + 1):
    train_metrics = train_last_block_epoch(
        model_to_train=last_block_model,
        loader=last_block_train_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        trainable_parameters=trainable_parameters,
    )

    val_metrics, _ = evaluate_hr_model(
        model=last_block_model,
        loader=last_block_val_loader,
        device=DEVICE,
    )

    current_val_mae = float(val_metrics["mae_mean"])
    improved = current_val_mae < (best_val_mae - LAST_BLOCK_MIN_DELTA)

    if improved:
        best_val_mae = current_val_mae
        best_epoch = epoch
        best_state = copy.deepcopy(last_block_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["train_loss"],
            "train_mae": train_metrics["train_mae"],
            "val_mae": current_val_mae,
            "val_median": float(val_metrics["mae_median"]),
            "val_p90": float(val_metrics["mae_p90"]),
            "grad_norm_mean": train_metrics["grad_norm_mean"],
            "grad_norm_max": train_metrics["grad_norm_max"],
            "improved": improved,
            "best_epoch_so_far": best_epoch,
            "best_val_mae_so_far": best_val_mae,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train MAE {train_metrics['train_mae']:.4f} | "
        f"val MAE {current_val_mae:.4f} | "
        f"best {best_val_mae:.4f} @ epoch {best_epoch}"
    )

    if epochs_without_improvement >= LAST_BLOCK_PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

last_block_history = pd.DataFrame(history_rows)

last_block_model.load_state_dict(best_state)
last_block_model.eval()

last_block_eval_metrics = {}
last_block_eval_predictions = {}

for role, loader in cached_loaders.items():
    metrics, predictions_df = evaluate_hr_model(
        model=last_block_model,
        loader=loader,
        device=DEVICE,
    )
    last_block_eval_metrics[role] = metrics
    last_block_eval_predictions[role] = predictions_df

last_block_comparison = compare_eval_to_frozen(
    candidate_metrics=last_block_eval_metrics,
    frozen_metrics=frozen_eval_metrics,
)

last_block_by_dataset_role = []

for role, predictions_df in last_block_eval_predictions.items():
    grouped = (
        predictions_df
        .groupby(["dataset", "window_role"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            target_hr_mean=("target_hr", "mean"),
        )
        .reset_index()
    )
    last_block_by_dataset_role.append(grouped)

last_block_by_dataset_role = pd.concat(last_block_by_dataset_role, axis=0).reset_index(drop=True)

for col in ["mae_mean", "mae_median", "mae_p90", "target_hr_mean"]:
    last_block_by_dataset_role[col] = last_block_by_dataset_role[col].round(4)

history_path = LAST_BLOCK_DIR / "last_block_head_sourcefps_history.csv"
comparison_path = LAST_BLOCK_DIR / "last_block_head_sourcefps_comparison.csv"
dataset_role_path = LAST_BLOCK_DIR / "last_block_head_sourcefps_by_dataset_role.csv"
checkpoint_path = LAST_BLOCK_DIR / "last_block_head_sourcefps_best.pt"

last_block_history.to_csv(history_path, index=False)
last_block_comparison.to_csv(comparison_path, index=False)
last_block_by_dataset_role.to_csv(dataset_role_path, index=False)

torch.save(
    {
        "experiment": "last_block_head_sourcefps_live_compatible",
        "base_checkpoint_path": str(CHECKPOINT_PATH),
        "trainable_mode": "last_transformer_block_plus_head",
        "model_state": last_block_model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_mae": best_val_mae,
        "frozen_val_mae": frozen_val_mae,
        "history": last_block_history.to_dict("records"),
        "comparison": last_block_comparison.to_dict("records"),
        "by_dataset_role": last_block_by_dataset_role.to_dict("records"),
        "config": {
            "epochs": LAST_BLOCK_EPOCHS,
            "batch_size": LAST_BLOCK_BATCH_SIZE,
            "last_block_lr": LAST_BLOCK_LR,
            "head_lr": LAST_BLOCK_HEAD_LR,
            "weight_decay": LAST_BLOCK_WEIGHT_DECAY,
            "patience": LAST_BLOCK_PATIENCE,
            "min_delta": LAST_BLOCK_MIN_DELTA,
            "huber_delta": float(best_params["huber_delta"]),
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "checkpoint_metadata": {
            "input_mode": checkpoint.get("input_mode"),
            "in_channels": checkpoint.get("in_channels"),
            "best_params": checkpoint.get("best_params"),
        },
    },
    checkpoint_path,
)

print("\nLast-block plus head training history:")
display(last_block_history.round(4))

print("\nLast-block plus head vs frozen comparison:")
display(last_block_comparison)

print("\nLast-block plus head by dataset and role:")
display(last_block_by_dataset_role.sort_values(["window_role", "dataset"]))

val_delta = float(
    last_block_comparison.loc[
        last_block_comparison["role"] == "finetune_val",
        "delta_mae_mean",
    ].iloc[0]
)

test_delta = float(
    last_block_comparison.loc[
        last_block_comparison["role"] == "finetune_test",
        "delta_mae_mean",
    ].iloc[0]
)

ood_delta = float(
    last_block_comparison.loc[
        last_block_comparison["role"] == "ood_eval",
        "delta_mae_mean",
    ].iloc[0]
)

if best_epoch == 0:
    decision = "No meaningful validation improvement. Do not adopt this checkpoint."
elif val_delta <= -0.10 and test_delta <= 0.10:
    decision = "Candidate improvement. Review dataset-level and OOD behavior before adopting."
else:
    decision = "Validation changed, but generalization is not clearly improved. Do not adopt without further analysis."

result_summary = {
    "best_epoch": int(best_epoch),
    "frozen_val_mae": float(frozen_val_mae),
    "best_val_mae": float(best_val_mae),
    "val_delta_vs_frozen": val_delta,
    "test_delta_vs_frozen": test_delta,
    "ood_delta_vs_frozen": ood_delta,
    "decision": decision,
    "history_path": str(history_path),
    "comparison_path": str(comparison_path),
    "dataset_role_path": str(dataset_role_path),
    "checkpoint_path": str(checkpoint_path),
}

print("\nLast-block plus head result summary:")
print(json.dumps(result_summary, indent=2))

print("\nDONE: Last-block plus head fine-tuning experiment completed.")

Last-block plus head parameter summary:


,module,trainable,parameters
0,encoder,False,1696
1,freq_proj,False,47680
2,head,True,2625
3,transformer_layers,False,202608
4,transformer_layers,True,67536


Frozen validation MAE baseline: 6.0275 BPM
Epoch 01 | train MAE 8.6542 | val MAE 6.0362 | best 6.0275 @ epoch 0
Epoch 02 | train MAE 8.6737 | val MAE 5.9972 | best 5.9972 @ epoch 2
Epoch 03 | train MAE 8.6483 | val MAE 6.1260 | best 5.9972 @ epoch 2
Epoch 04 | train MAE 8.5337 | val MAE 6.0730 | best 5.9972 @ epoch 2
Epoch 05 | train MAE 8.6282 | val MAE 6.1018 | best 5.9972 @ epoch 2
Epoch 06 | train MAE 8.7135 | val MAE 6.0400 | best 5.9972 @ epoch 2
Epoch 07 | train MAE 8.6131 | val MAE 6.0651 | best 5.9972 @ epoch 2
Epoch 08 | train MAE 8.6387 | val MAE 6.0791 | best 5.9972 @ epoch 2
Epoch 09 | train MAE 8.5552 | val MAE 6.0804 | best 5.9972 @ epoch 2
Early stopping after 9 epochs.

Last-block plus head training history:


,epoch,train_loss,train_mae,val_mae,val_median,val_p90,grad_norm_mean,grad_norm_max,improved,best_epoch_so_far,best_val_mae_so_far
0,1,38.8990,8.6542,6.0362,4.0417,14.3236,117.4038,327.4935,False,0,6.0275
1,2,39.0197,8.6737,5.9972,4.0256,14.3770,82.5452,223.1174,True,2,5.9972
2,3,38.8195,8.6483,6.1260,4.1164,14.3519,88.9573,232.8523,False,2,5.9972
3,4,38.1957,8.5337,6.0730,4.0779,14.3343,63.4699,180.8870,False,2,5.9972
4,5,38.7162,8.6282,6.1018,4.1031,14.3369,80.1082,204.3141,False,2,5.9972
5,6,39.2490,8.7135,6.0400,4.0742,14.3335,84.1480,247.9008,False,2,5.9972
6,7,38.6310,8.6131,6.0651,4.1021,14.3336,81.3266,199.8792,False,2,5.9972
7,8,38.7510,8.6387,6.0791,4.1066,14.3031,79.4378,220.5081,False,2,5.9972
8,9,38.3504,8.5552,6.0804,4.1194,14.2805,90.6762,254.4738,False,2,5.9972



Last-block plus head vs frozen comparison:


,role,rows,frozen_mae_mean,candidate_mae_mean,delta_mae_mean,frozen_mae_median,candidate_mae_median,delta_mae_median,frozen_mae_p90,candidate_mae_p90,delta_mae_p90
0,finetune_train,12503,5.8236,5.8266,0.0030,3.7437,3.9408,0.1971,13.7192,13.4796,-0.2396
1,finetune_val,2839,6.0275,5.9972,-0.0303,3.8533,4.0256,0.1723,14.0701,14.3770,0.3069
2,finetune_test,2828,5.7696,5.7845,0.0149,3.6274,3.6699,0.0425,13.6984,13.2907,-0.4076
3,ood_eval,882,14.0355,14.6563,0.6208,7.6838,7.8609,0.1770,38.1498,40.5404,2.3906



Last-block plus head by dataset and role:


,dataset,window_role,rows,mae_mean,mae_median,mae_p90,target_hr_mean
6,mcd_rppg,finetune_test,2094,4.9014,3.2390,11.2694,79.8533
7,ubfc_phys,finetune_test,658,8.7913,6.0576,19.1077,77.4205
8,ubfc_rppg,finetune_test,76,4.0843,3.4585,7.9783,98.2693
0,mcd_rppg,finetune_train,7446,4.6356,3.1849,10.1703,79.0648
1,ubfc_phys,finetune_train,4595,7.7813,5.6656,17.7068,80.3055
2,ubfc_rppg,finetune_train,462,5.5812,4.3475,11.7705,98.4492
3,mcd_rppg,finetune_val,1743,4.9275,3.2402,11.0662,79.1086
4,ubfc_phys,finetune_val,986,7.9023,5.5319,17.8224,80.0024
5,ubfc_rppg,finetune_val,110,5.8701,3.8611,14.8821,98.2772
9,ecg_fitness,ood_eval,882,14.6563,7.8609,40.5404,109.2048



Last-block plus head result summary:
{
  "best_epoch": 2,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 5.997190952301025,
  "val_delta_vs_frozen": -0.0303,
  "test_delta_vs_frozen": 0.0149,
  "ood_delta_vs_frozen": 0.6208,
  "decision": "Validation changed, but generalization is not clearly improved. Do not adopt without further analysis.",
  "history_path": "/kaggle/working/crvse_last_block_head_sourcefps/last_block_head_sourcefps_history.csv",
  "comparison_path": "/kaggle/working/crvse_last_block_head_sourcefps/last_block_head_sourcefps_comparison.csv",
  "dataset_role_path": "/kaggle/working/crvse_last_block_head_sourcefps/last_block_head_sourcefps_by_dataset_role.csv",
  "checkpoint_path": "/kaggle/working/crvse_last_block_head_sourcefps/last_block_head_sourcefps_best.pt"
}

DONE: Last-block plus head fine-tuning experiment completed.


## Balanced Dataset Sampling For Fine-Tuning

The previous fine-tuning experiments produced only tiny validation improvements and worsened some smaller or out-of-domain groups.

This section tests whether the problem is partly caused by dataset imbalance.

The training set is dominated by:

```text
MCD-rPPG
UBFC-Phys

In [16]:
BALANCED_EPOCHS = 30
BALANCED_BATCH_SIZE = 256
BALANCED_LAST_BLOCK_LR = 1e-5
BALANCED_HEAD_LR = 5e-5
BALANCED_WEIGHT_DECAY = 1e-4
BALANCED_PATIENCE = 7
BALANCED_MIN_DELTA = 0.02

BALANCED_DIR = KAGGLE_WORKING / "crvse_balanced_last_block_head_sourcefps"
BALANCED_DIR.mkdir(parents=True, exist_ok=True)


def make_dataset_balanced_sampler(dataset: MaterializedTensorDataset) -> WeightedRandomSampler:
    """Create a sampler that approximately balances training windows by dataset."""
    metadata = dataset.metadata.copy()

    dataset_counts = metadata["dataset"].value_counts().to_dict()

    sample_weights = metadata["dataset"].map(
        lambda dataset_name: 1.0 / dataset_counts[dataset_name]
    ).astype(np.float64).to_numpy()

    sample_weights = torch.tensor(sample_weights, dtype=torch.double)

    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_balanced_cached_loader(
    dataset: MaterializedTensorDataset,
    batch_size: int,
) -> DataLoader:
    """Create a balanced DataLoader using dataset-level inverse-frequency sampling."""
    sampler = make_dataset_balanced_sampler(dataset)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_with_metadata,
        pin_memory=(DEVICE.type == "cuda"),
    )


balanced_train_loader = make_balanced_cached_loader(
    train_cached_ds,
    batch_size=BALANCED_BATCH_SIZE,
)

balanced_val_loader = make_cached_loader(
    val_cached_ds,
    batch_size=BALANCED_BATCH_SIZE,
    shuffle=False,
)

# Inspect the first few balanced batches to confirm that sampling changed.
balanced_batch_rows = []

for batch_index, (_x, _y, metadata) in enumerate(balanced_train_loader):
    for row in metadata:
        balanced_batch_rows.append(
            {
                "batch": batch_index,
                "dataset": row["dataset"],
                "window_role": row["window_role"],
            }
        )

    if batch_index >= 4:
        break

balanced_batch_preview = pd.DataFrame(balanced_batch_rows)

print("Original train rows by dataset:")
display(
    train_cached_ds.metadata
    .groupby("dataset")
    .size()
    .reset_index(name="rows")
    .sort_values("dataset")
)

print("\nBalanced sampler preview from first 5 batches:")
display(
    balanced_batch_preview
    .groupby(["batch", "dataset"])
    .size()
    .reset_index(name="rows")
    .sort_values(["batch", "dataset"])
)


balanced_model = copy.deepcopy(model).to(DEVICE)
trainable_parameters = freeze_for_last_block_plus_head(balanced_model)

last_block_params = list(balanced_model.transformer_layers[-1].parameters())
head_params = list(balanced_model.head.parameters())

optimizer = AdamW(
    [
        {"params": last_block_params, "lr": BALANCED_LAST_BLOCK_LR},
        {"params": head_params, "lr": BALANCED_HEAD_LR},
    ],
    weight_decay=BALANCED_WEIGHT_DECAY,
)

loss_fn = nn.HuberLoss(delta=float(best_params["huber_delta"]))

frozen_val_mae = float(frozen_eval_metrics["finetune_val"]["mae_mean"])
best_val_mae = frozen_val_mae
best_epoch = 0
best_state = copy.deepcopy(balanced_model.state_dict())
epochs_without_improvement = 0
history_rows = []

print("\nBalanced last-block plus head parameter summary:")
display(parameter_summary(balanced_model))

print(f"Frozen validation MAE baseline: {frozen_val_mae:.4f} BPM")

for epoch in range(1, BALANCED_EPOCHS + 1):
    train_metrics = train_last_block_epoch(
        model_to_train=balanced_model,
        loader=balanced_train_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        trainable_parameters=trainable_parameters,
    )

    val_metrics, _ = evaluate_hr_model(
        model=balanced_model,
        loader=balanced_val_loader,
        device=DEVICE,
    )

    current_val_mae = float(val_metrics["mae_mean"])
    improved = current_val_mae < (best_val_mae - BALANCED_MIN_DELTA)

    if improved:
        best_val_mae = current_val_mae
        best_epoch = epoch
        best_state = copy.deepcopy(balanced_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["train_loss"],
            "train_mae": train_metrics["train_mae"],
            "val_mae": current_val_mae,
            "val_median": float(val_metrics["mae_median"]),
            "val_p90": float(val_metrics["mae_p90"]),
            "grad_norm_mean": train_metrics["grad_norm_mean"],
            "grad_norm_max": train_metrics["grad_norm_max"],
            "improved": improved,
            "best_epoch_so_far": best_epoch,
            "best_val_mae_so_far": best_val_mae,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train MAE {train_metrics['train_mae']:.4f} | "
        f"val MAE {current_val_mae:.4f} | "
        f"best {best_val_mae:.4f} @ epoch {best_epoch}"
    )

    if epochs_without_improvement >= BALANCED_PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

balanced_history = pd.DataFrame(history_rows)

balanced_model.load_state_dict(best_state)
balanced_model.eval()

balanced_eval_metrics = {}
balanced_eval_predictions = {}

for role, loader in cached_loaders.items():
    metrics, predictions_df = evaluate_hr_model(
        model=balanced_model,
        loader=loader,
        device=DEVICE,
    )
    balanced_eval_metrics[role] = metrics
    balanced_eval_predictions[role] = predictions_df

balanced_comparison = compare_eval_to_frozen(
    candidate_metrics=balanced_eval_metrics,
    frozen_metrics=frozen_eval_metrics,
)

balanced_by_dataset_role = []

for role, predictions_df in balanced_eval_predictions.items():
    grouped = (
        predictions_df
        .groupby(["dataset", "window_role"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            target_hr_mean=("target_hr", "mean"),
        )
        .reset_index()
    )
    balanced_by_dataset_role.append(grouped)

balanced_by_dataset_role = pd.concat(balanced_by_dataset_role, axis=0).reset_index(drop=True)

for col in ["mae_mean", "mae_median", "mae_p90", "target_hr_mean"]:
    balanced_by_dataset_role[col] = balanced_by_dataset_role[col].round(4)

history_path = BALANCED_DIR / "balanced_last_block_head_sourcefps_history.csv"
comparison_path = BALANCED_DIR / "balanced_last_block_head_sourcefps_comparison.csv"
dataset_role_path = BALANCED_DIR / "balanced_last_block_head_sourcefps_by_dataset_role.csv"
checkpoint_path = BALANCED_DIR / "balanced_last_block_head_sourcefps_best.pt"

balanced_history.to_csv(history_path, index=False)
balanced_comparison.to_csv(comparison_path, index=False)
balanced_by_dataset_role.to_csv(dataset_role_path, index=False)

torch.save(
    {
        "experiment": "balanced_last_block_head_sourcefps_live_compatible",
        "base_checkpoint_path": str(CHECKPOINT_PATH),
        "trainable_mode": "last_transformer_block_plus_head",
        "sampler": "inverse_dataset_frequency",
        "model_state": balanced_model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_mae": best_val_mae,
        "frozen_val_mae": frozen_val_mae,
        "history": balanced_history.to_dict("records"),
        "comparison": balanced_comparison.to_dict("records"),
        "by_dataset_role": balanced_by_dataset_role.to_dict("records"),
        "config": {
            "epochs": BALANCED_EPOCHS,
            "batch_size": BALANCED_BATCH_SIZE,
            "last_block_lr": BALANCED_LAST_BLOCK_LR,
            "head_lr": BALANCED_HEAD_LR,
            "weight_decay": BALANCED_WEIGHT_DECAY,
            "patience": BALANCED_PATIENCE,
            "min_delta": BALANCED_MIN_DELTA,
            "huber_delta": float(best_params["huber_delta"]),
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "checkpoint_metadata": {
            "input_mode": checkpoint.get("input_mode"),
            "in_channels": checkpoint.get("in_channels"),
            "best_params": checkpoint.get("best_params"),
        },
    },
    checkpoint_path,
)

print("\nBalanced training history:")
display(balanced_history.round(4))

print("\nBalanced last-block plus head vs frozen comparison:")
display(balanced_comparison)

print("\nBalanced last-block plus head by dataset and role:")
display(balanced_by_dataset_role.sort_values(["window_role", "dataset"]))

val_delta = float(
    balanced_comparison.loc[
        balanced_comparison["role"] == "finetune_val",
        "delta_mae_mean",
    ].iloc[0]
)

test_delta = float(
    balanced_comparison.loc[
        balanced_comparison["role"] == "finetune_test",
        "delta_mae_mean",
    ].iloc[0]
)

ood_delta = float(
    balanced_comparison.loc[
        balanced_comparison["role"] == "ood_eval",
        "delta_mae_mean",
    ].iloc[0]
)

if best_epoch == 0:
    decision = "No meaningful validation improvement. Do not adopt this checkpoint."
elif val_delta <= -0.10 and test_delta <= 0.10:
    decision = "Candidate improvement. Review dataset-level and OOD behavior before adopting."
else:
    decision = "Balanced sampling did not produce clear generalization improvement. Do not adopt without further analysis."

balanced_result_summary = {
    "best_epoch": int(best_epoch),
    "frozen_val_mae": float(frozen_val_mae),
    "best_val_mae": float(best_val_mae),
    "val_delta_vs_frozen": val_delta,
    "test_delta_vs_frozen": test_delta,
    "ood_delta_vs_frozen": ood_delta,
    "decision": decision,
    "history_path": str(history_path),
    "comparison_path": str(comparison_path),
    "dataset_role_path": str(dataset_role_path),
    "checkpoint_path": str(checkpoint_path),
}

print("\nBalanced result summary:")
print(json.dumps(balanced_result_summary, indent=2))

print("\nDONE: Balanced last-block plus head fine-tuning experiment completed.")

Original train rows by dataset:


,dataset,rows
0,mcd_rppg,7446
1,ubfc_phys,4595
2,ubfc_rppg,462



Balanced sampler preview from first 5 batches:


,batch,dataset,rows
0,0,mcd_rppg,83
1,0,ubfc_phys,98
2,0,ubfc_rppg,75
3,1,mcd_rppg,90
4,1,ubfc_phys,77
5,1,ubfc_rppg,89
6,2,mcd_rppg,87
7,2,ubfc_phys,82
8,2,ubfc_rppg,87
9,3,mcd_rppg,82



Balanced last-block plus head parameter summary:


...


Balanced last-block plus head vs frozen comparison:


,role,rows,frozen_mae_mean,candidate_mae_mean,delta_mae_mean,frozen_mae_median,candidate_mae_median,delta_mae_median,frozen_mae_p90,candidate_mae_p90,delta_mae_p90
0,finetune_train,12503,5.8236,5.7720,-0.0516,3.7437,3.8086,0.0649,13.7192,13.5059,-0.2133
1,finetune_val,2839,6.0275,5.9516,-0.0759,3.8533,3.9106,0.0573,14.0701,14.3093,0.2391
2,finetune_test,2828,5.7696,5.7349,-0.0347,3.6274,3.5666,-0.0608,13.6984,13.1621,-0.5363
3,ood_eval,882,14.0355,14.4675,0.4320,7.6838,7.9843,0.3005,38.1498,39.9524,1.8026



Balanced last-block plus head by dataset and role:


,dataset,window_role,rows,mae_mean,mae_median,mae_p90,target_hr_mean
6,mcd_rppg,finetune_test,2094,4.8464,3.1478,11.0913,79.8533
7,ubfc_phys,finetune_test,658,8.7883,6.0568,19.1163,77.4205
8,ubfc_rppg,finetune_test,76,3.7783,3.4204,7.2969,98.2693
0,mcd_rppg,finetune_train,7446,4.5882,3.1393,10.1612,79.0648
1,ubfc_phys,finetune_train,4595,7.7430,5.5637,17.9793,80.3055
2,ubfc_rppg,finetune_train,462,5.2475,3.9585,11.2277,98.4492
3,mcd_rppg,finetune_val,1743,4.9170,3.2860,11.1629,79.1086
4,ubfc_phys,finetune_val,986,7.8392,5.4013,18.1722,80.0024
5,ubfc_rppg,finetune_val,110,5.4253,3.6115,14.2104,98.2772
9,ecg_fitness,ood_eval,882,14.4675,7.9843,39.9525,109.2048



Balanced result summary:
{
  "best_epoch": 2,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 5.951594829559326,
  "val_delta_vs_frozen": -0.0759,
  "test_delta_vs_frozen": -0.0347,
  "ood_delta_vs_frozen": 0.432,
  "decision": "Balanced sampling did not produce clear generalization improvement. Do not adopt without further analysis.",
  "history_path": "/kaggle/working/crvse_balanced_last_block_head_sourcefps/balanced_last_block_head_sourcefps_history.csv",
  "comparison_path": "/kaggle/working/crvse_balanced_last_block_head_sourcefps/balanced_last_block_head_sourcefps_comparison.csv",
  "dataset_role_path": "/kaggle/working/crvse_balanced_last_block_head_sourcefps/balanced_last_block_head_sourcefps_by_dataset_role.csv",
  "checkpoint_path": "/kaggle/working/crvse_balanced_last_block_head_sourcefps/balanced_last_block_head_sourcefps_best.pt"
}

DONE: Balanced last-block plus head fine-tuning experiment completed.


## Fine-Tuning Experiment Scoreboard

This section compares all fine-tuning attempts in this notebook.

The goal is to decide whether any checkpoint should replace the frozen model.

Experiments compared:

- frozen source-FPS baseline,
- head-only fine-tuning,
- last-block plus head fine-tuning,
- balanced last-block plus head fine-tuning.

A checkpoint should not be adopted just because validation mean MAE improves slightly.

A useful checkpoint should improve validation, remain stable on held-out test data, and avoid unacceptable degradation on ECG-Fitness out-of-domain evaluation.

In [17]:
def normalize_comparison_table(
    comparison: pd.DataFrame,
    experiment_name: str,
) -> pd.DataFrame:
    """Return a comparison table with consistent candidate metric column names."""
    table = comparison.copy()

    candidate_prefixes = [
        "candidate",
        "head_only",
        "last_block",
        "balanced",
    ]

    rename_map = {}

    for prefix in candidate_prefixes:
        for metric in ["mean", "median", "p90"]:
            source_col = f"{prefix}_mae_{metric}"
            target_col = f"candidate_mae_{metric}"

            if source_col in table.columns and target_col not in table.columns:
                rename_map[source_col] = target_col

    table = table.rename(columns=rename_map)
    table["experiment"] = experiment_name

    required_cols = [
        "experiment",
        "role",
        "rows",
        "frozen_mae_mean",
        "candidate_mae_mean",
        "delta_mae_mean",
        "frozen_mae_median",
        "candidate_mae_median",
        "delta_mae_median",
        "frozen_mae_p90",
        "candidate_mae_p90",
        "delta_mae_p90",
    ]

    missing_cols = [col for col in required_cols if col not in table.columns]

    if missing_cols:
        raise KeyError(
            f"{experiment_name} comparison is missing columns: {missing_cols}\n"
            f"Available columns: {list(table.columns)}"
        )

    return table[required_cols]


scoreboard = pd.concat(
    [
        normalize_comparison_table(head_only_comparison, "head_only"),
        normalize_comparison_table(last_block_comparison, "last_block_head"),
        normalize_comparison_table(balanced_comparison, "balanced_last_block_head"),
    ],
    axis=0,
).reset_index(drop=True)

scoreboard_metric_cols = [
    "frozen_mae_mean",
    "candidate_mae_mean",
    "delta_mae_mean",
    "frozen_mae_median",
    "candidate_mae_median",
    "delta_mae_median",
    "frozen_mae_p90",
    "candidate_mae_p90",
    "delta_mae_p90",
]

scoreboard[scoreboard_metric_cols] = scoreboard[scoreboard_metric_cols].round(4)

print("Fine-tuning experiment scoreboard:")
display(scoreboard.sort_values(["role", "experiment"]))


decision_rows = []

for experiment_name, experiment_table in scoreboard.groupby("experiment"):
    by_role = experiment_table.set_index("role")

    val_delta = float(by_role.loc["finetune_val", "delta_mae_mean"])
    test_delta = float(by_role.loc["finetune_test", "delta_mae_mean"])
    ood_delta = float(by_role.loc["ood_eval", "delta_mae_mean"])

    val_p90_delta = float(by_role.loc["finetune_val", "delta_mae_p90"])
    test_p90_delta = float(by_role.loc["finetune_test", "delta_mae_p90"])

    adoption_status = "reject"

    if val_delta <= -0.10 and test_delta <= 0.05 and ood_delta <= 0.25:
        adoption_status = "candidate"
    elif val_delta < 0 and test_delta <= 0:
        adoption_status = "weak_candidate_for_analysis_only"

    decision_rows.append(
        {
            "experiment": experiment_name,
            "val_delta_mean": val_delta,
            "test_delta_mean": test_delta,
            "ood_delta_mean": ood_delta,
            "val_delta_p90": val_p90_delta,
            "test_delta_p90": test_p90_delta,
            "adoption_status": adoption_status,
        }
    )

decision_table = pd.DataFrame(decision_rows).sort_values("val_delta_mean")

for col in [
    "val_delta_mean",
    "test_delta_mean",
    "ood_delta_mean",
    "val_delta_p90",
    "test_delta_p90",
]:
    decision_table[col] = decision_table[col].round(4)

print("\nExperiment decision table:")
display(decision_table)


nb_p2_10_conclusion = {
    "best_aggregate_experiment": "balanced_last_block_head",
    "best_val_delta_bpm": float(
        decision_table.loc[
            decision_table["experiment"] == "balanced_last_block_head",
            "val_delta_mean",
        ].iloc[0]
    ),
    "best_test_delta_bpm": float(
        decision_table.loc[
            decision_table["experiment"] == "balanced_last_block_head",
            "test_delta_mean",
        ].iloc[0]
    ),
    "best_ood_delta_bpm": float(
        decision_table.loc[
            decision_table["experiment"] == "balanced_last_block_head",
            "ood_delta_mean",
        ].iloc[0]
    ),
    "adopt_checkpoint": False,
    "reason": (
        "Fine-tuning produced only small aggregate improvements. "
        "Balanced last-block/head was the best variant, but the gain was below "
        "the practical threshold and ECG-Fitness OOD performance degraded. "
        "No fine-tuned checkpoint should replace the frozen model from this notebook."
    ),
    "next_research_direction": (
        "If improvement is still desired, the next notebook should test training "
        "or transfer learning on live-compatible tensors as the primary training "
        "distribution, rather than shallow adaptation of the old checkpoint."
    ),
}

print("\nNB_P2_10 conclusion:")
print(json.dumps(nb_p2_10_conclusion, indent=2))

Fine-tuning experiment scoreboard:


,experiment,role,rows,frozen_mae_mean,candidate_mae_mean,delta_mae_mean,frozen_mae_median,candidate_mae_median,delta_mae_median,frozen_mae_p90,candidate_mae_p90,delta_mae_p90
10,balanced_last_block_head,finetune_test,2828,5.7696,5.7349,-0.0347,3.6274,3.5666,-0.0608,13.6984,13.1621,-0.5363
2,head_only,finetune_test,2828,5.7696,5.7841,0.0144,3.6274,3.6789,0.0516,13.6984,13.3458,-0.3526
6,last_block_head,finetune_test,2828,5.7696,5.7845,0.0149,3.6274,3.6699,0.0425,13.6984,13.2907,-0.4076
8,balanced_last_block_head,finetune_train,12503,5.8236,5.7720,-0.0516,3.7437,3.8086,0.0649,13.7192,13.5059,-0.2133
0,head_only,finetune_train,12503,5.8236,5.8359,0.0123,3.7437,3.9703,0.2266,13.7192,13.5125,-0.2067
4,last_block_head,finetune_train,12503,5.8236,5.8266,0.0030,3.7437,3.9408,0.1971,13.7192,13.4796,-0.2396
9,balanced_last_block_head,finetune_val,2839,6.0275,5.9516,-0.0759,3.8533,3.9106,0.0573,14.0701,14.3093,0.2391
1,head_only,finetune_val,2839,6.0275,6.0047,-0.0228,3.8533,4.0423,0.1890,14.0701,14.3648,0.2947
5,last_block_head,finetune_val,2839,6.0275,5.9972,-0.0303,3.8533,4.0256,0.1723,14.0701,14.3770,0.3069
11,balanced_last_block_head,ood_eval,882,14.0355,14.4675,0.4320,7.6838,7.9843,0.3005,38.1498,39.9524,1.8026



Experiment decision table:


,experiment,val_delta_mean,test_delta_mean,ood_delta_mean,val_delta_p90,test_delta_p90,adoption_status
0,balanced_last_block_head,-0.0759,-0.0347,0.4320,0.2391,-0.5363,weak_candidate_for_analysis_only
2,last_block_head,-0.0303,0.0149,0.6208,0.3069,-0.4076,reject
1,head_only,-0.0228,0.0144,0.6904,0.2947,-0.3526,reject



NB_P2_10 conclusion:
{
  "best_aggregate_experiment": "balanced_last_block_head",
  "best_val_delta_bpm": -0.0759,
  "best_test_delta_bpm": -0.0347,
  "best_ood_delta_bpm": 0.432,
  "adopt_checkpoint": false,
  "reason": "Fine-tuning produced only small aggregate improvements. Balanced last-block/head was the best variant, but the gain was below the practical threshold and ECG-Fitness OOD performance degraded. No fine-tuned checkpoint should replace the frozen model from this notebook.",
  "next_research_direction": "If improvement is still desired, the next notebook should test training or transfer learning on live-compatible tensors as the primary training distribution, rather than shallow adaptation of the old checkpoint."
}


In [18]:
NB_P2_10_SUMMARY_DIR = KAGGLE_WORKING / "crvse_nb_p2_10_summary"
NB_P2_10_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

scoreboard_path = NB_P2_10_SUMMARY_DIR / "nb_p2_10_finetuning_scoreboard.csv"
decision_table_path = NB_P2_10_SUMMARY_DIR / "nb_p2_10_decision_table.csv"
conclusion_json_path = NB_P2_10_SUMMARY_DIR / "nb_p2_10_conclusion.json"
conclusion_md_path = NB_P2_10_SUMMARY_DIR / "nb_p2_10_conclusion.md"

scoreboard.to_csv(scoreboard_path, index=False)
decision_table.to_csv(decision_table_path, index=False)

with conclusion_json_path.open("w", encoding="utf-8") as file:
    json.dump(nb_p2_10_conclusion, file, indent=2)

best_row = decision_table.loc[
    decision_table["experiment"] == nb_p2_10_conclusion["best_aggregate_experiment"]
].iloc[0]

conclusion_md = f"""# NB_P2_10 Live-Compatible Fine-Tuning Conclusion

## Best Experiment

`{nb_p2_10_conclusion["best_aggregate_experiment"]}`

## Result Versus Frozen Source-FPS Baseline

| Metric | Delta BPM |
|---|---:|
| Validation mean MAE | {best_row["val_delta_mean"]:.4f} |
| Test mean MAE | {best_row["test_delta_mean"]:.4f} |
| ECG-Fitness OOD mean MAE | {best_row["ood_delta_mean"]:.4f} |
| Validation p90 MAE | {best_row["val_delta_p90"]:.4f} |
| Test p90 MAE | {best_row["test_delta_p90"]:.4f} |

## Adoption Decision

Adopt fine-tuned checkpoint: **{nb_p2_10_conclusion["adopt_checkpoint"]}**

## Reason

{nb_p2_10_conclusion["reason"]}

## Next Research Direction

{nb_p2_10_conclusion["next_research_direction"]}

## Scientific Interpretation

The shallow fine-tuning experiments produced only small aggregate improvements.
The best variant was balanced last-block/head fine-tuning, but the gain was below
a practical threshold and ECG-Fitness out-of-domain performance degraded.

This supports the conclusion that the current checkpoint does not adapt strongly
enough through shallow fine-tuning alone. A future notebook should test training
or transfer learning with live-compatible tensors as the primary training
distribution.
"""

conclusion_md_path.write_text(conclusion_md, encoding="utf-8")

saved_artifacts = pd.DataFrame(
    [
        {"artifact": "scoreboard_csv", "path": str(scoreboard_path)},
        {"artifact": "decision_table_csv", "path": str(decision_table_path)},
        {"artifact": "conclusion_json", "path": str(conclusion_json_path)},
        {"artifact": "conclusion_markdown", "path": str(conclusion_md_path)},
    ]
)

print("Saved NB_P2_10 conclusion artifacts:")
display(saved_artifacts)

print("\nConclusion markdown preview:")
print(conclusion_md)

print("\nPASS: NB_P2_10 conclusion artifacts saved.")

Saved NB_P2_10 conclusion artifacts:


,artifact,path
0,scoreboard_csv,/kaggle/working/crvse_nb_p2_10_summary/nb_p2_1...
1,decision_table_csv,/kaggle/working/crvse_nb_p2_10_summary/nb_p2_1...
2,conclusion_json,/kaggle/working/crvse_nb_p2_10_summary/nb_p2_1...
3,conclusion_markdown,/kaggle/working/crvse_nb_p2_10_summary/nb_p2_1...



Conclusion markdown preview:
# NB_P2_10 Live-Compatible Fine-Tuning Conclusion

## Best Experiment

`balanced_last_block_head`

## Result Versus Frozen Source-FPS Baseline

| Metric | Delta BPM |
|---|---:|
| Validation mean MAE | -0.0759 |
| Test mean MAE | -0.0347 |
| ECG-Fitness OOD mean MAE | 0.4320 |
| Validation p90 MAE | 0.2391 |
| Test p90 MAE | -0.5363 |

## Adoption Decision

Adopt fine-tuned checkpoint: **False**

## Reason

Fine-tuning produced only small aggregate improvements. Balanced last-block/head was the best variant, but the gain was below the practical threshold and ECG-Fitness OOD performance degraded. No fine-tuned checkpoint should replace the frozen model from this notebook.

## Next Research Direction

If improvement is still desired, the next notebook should test training or transfer learning on live-compatible tensors as the primary training distribution, rather than shallow adaptation of the old checkpoint.

## Scientific Interpretation

The shallow fine-t